# 1. 파일 확인(이미지 파일 갯수)

In [1]:
# Cell 1: image-only input (scan images)

from pathlib import Path
from tqdm.auto import tqdm

IMAGE_ROOT = Path("train_artwork")   # ✅ 너 환경에 맞게 여기만 수정

img_ext = {".png", ".jpg", ".jpeg", ".webp"}
image_paths = [p for p in IMAGE_ROOT.rglob("*") if p.suffix.lower() in img_ext]

if not image_paths:
    raise FileNotFoundError(
        f"이미지를 못 찾았어. IMAGE_ROOT={IMAGE_ROOT.resolve()} 아래에 png/jpg/jpeg/webp가 있는지 확인해줘."
    )

print(f"[Found images] {len(image_paths)}")
print("[Example]", image_paths[0])

# artwork 메타(입력 JSON 없이 자동 생성)
def infer_artist_id(p: Path) -> str:
    # 기본 규칙: '상위 폴더명'을 artist_id로 사용
    # (작가 폴더가 없다면 여기 규칙을 네 파일명/폴더 구조에 맞게 바꾸면 됨)
    return p.parent.name

artworks = []
for i, p in enumerate(image_paths):
    artworks.append({
        "artwork_id": f"artwork_{i:06d}",
        "artist_id": infer_artist_id(p),
        "image_path": str(p.as_posix()),
    })

print("[Sample artwork row]", artworks[0])

/home/j-i14e107/.conda/envs/ai_dev_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Found images] 27702
[Example] train_artwork/train_031_022.png
[Sample artwork row] {'artwork_id': 'artwork_000000', 'artist_id': 'train_artwork', 'image_path': 'train_artwork/train_031_022.png'}


# 2. 카테고리별 지정해주기

In [2]:
# Cell 2: load CLIP + build text embeddings

import numpy as np
import torch
from transformers import CLIPProcessor, CLIPModel

TOPK = 8
MIN_SCORE_TO_ADD = 0.20
REL_TO_TOP1 = 0.85
FORCE_TOP2_REL = 0.92

WIKIART_GENRES_68 = [
    "abstract","advertisement","allegorical_painting","animal_painting","animation","architecture",
    "artists_book","augmented_reality","battle_painting","bijinga","bird_and_flower_painting",
    "calligraphy","capriccio","caricature","cityscape","cloudscape","design","digital","figurative",
    "flower_painting","furniture","genre_painting","graffiti","history_painting","icon","illustration",
    "installation","interior","jewelry","landscape","literary_painting","manga","marina","miniature",
    "mobile","mosaic","mural","mythological_painting","nude_painting","object","ornament","panorama",
    "pastorale","performance","photo","pin_up","portrait","poster","quadratura","religious_painting",
    "sculpture","self_portrait","shan_shui","sketch_and_study","stabile","still_life","symbolic_painting",
    "tapestry","tessellation","trompe_loeil","tronie","urushi_e","utensil","vanitas","veduta","video",
    "wildlife_painting","yakusha_e",
]
EXTRA_40 = [
    "conceptual_art","generative_art","ai_art","net_art","data_art","glitch_art","pixel_art","voxel_art",
    "3d_render_art","projection_mapping","vr_art","light_art",
    "land_art","eco_art","bio_art","sound_art","social_practice","participatory_art","relational_aesthetics",
    "minimalism","pop_art","op_art","photorealism","hyperrealism",
    "cartoon_western","editorial_illustration","concept_art","matte_painting","character_design",
    "anime_key_visual","webtoon_manhwa_style","isometric_illustration","vector_flat","vaporwave_synthwave",
    "dansaekhwa","minhwa_hojakdo","minhwa_chaekgeori","minhwa_munjado","minhwa_hwajodo","k_contemporary_pop",
]

ALL_LABELS = WIKIART_GENRES_68 + EXTRA_40
assert len(ALL_LABELS) == 108

def idx_to_category(i0: int) -> str:
    return f"category{i0+1:03d}"

TEXT_PROMPTS = [f"a painting in the style of {lab.replace('_',' ')}" for lab in ALL_LABELS]

device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = (device == "cuda")
print("[CLIP] device:", device)

model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device).eval()
processor = CLIPProcessor.from_pretrained(model_name)

with torch.no_grad():
    text_in = processor(text=TEXT_PROMPTS, return_tensors="pt", padding=True, truncation=True).to(device)
    if use_fp16:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            text_feat = model.get_text_features(**text_in)
    else:
        text_feat = model.get_text_features(**text_in)

text_feat = text_feat / (text_feat.norm(dim=-1, keepdim=True) + 1e-12)
print("[OK] text_feat:", tuple(text_feat.shape))

[CLIP] device: cuda


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[OK] text_feat: (108, 512)


In [3]:
# Cell 3 : 카테고리 배치하기
from collections import defaultdict

# 카테고리별 저장 번호 카운터 (category_002 -> 1,2,3...)
cat_counter = defaultdict(int)

# (선택) 재실행해도 번호 이어가고 싶으면, 기존 파일을 보고 초기값 세팅
# 단, 파일명이 "category_002_###.ext" 형태일 때만 잘 작동
def init_counter_from_existing(split_dir: Path):
    for d in split_dir.iterdir():
        if not d.is_dir():
            continue
        mx = 0
        for f in d.glob(f"{d.name}_*"):
            # 예: category_002_004.jpg -> "004"
            stem = f.stem
            parts = stem.split("_")
            if len(parts) >= 3 and parts[-1].isdigit():
                mx = max(mx, int(parts[-1]))
        if mx > 0:
            cat_counter[d.name] = mx

# init_counter_from_existing(SPLIT_DIR)  # 원치 않으면 이 줄 주석 처리

# 3. 카테고리 별로 분류하기(CLiP 모델 사용)

In [4]:
# Cell 4 (REPLACE): CLIP Top1 ONLY (BATCH) + FILE OPS (Top1 only, rename)
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
import torch
import shutil
from collections import defaultdict

torch.set_grad_enabled(False)
model.eval()

# ✅ 속도 옵션 (L40S에서 도움됨)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# ✅ 배치 크기
BATCH_SIZE = 512

# ✅ 출력 폴더 (카테고리 폴더들이 여기 아래 생김)
OUT_DIR = Path.cwd() / "split_category"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("[mkdir]", OUT_DIR, "exists?:", OUT_DIR.exists())

# ✅ 파일 작업 모드
# - copy: 원본 보존(추천)
# - move: 원본 이동(원본 폴더에서 사라짐)
MODE = "copy"   # "copy" or "move"
DRY_RUN = False # True면 실제 파일 작업 안 함(테스트용)

bad = 0
placed = 0
missing = 0
errors = 0

results = []

# 카테고리별 연번(0001, 0002, ...)
cat_counter = defaultdict(int)

def load_rgb(path: Path):
    try:
        # context manager로 열어서 바로 RGB로 복제(핸들 누수 방지)
        with Image.open(path) as im:
            return im.convert("RGB")
    except Exception:
        return None

def safe_name(s: str) -> str:
    return "".join(c for c in s if c not in r'\/:*?"<>|').strip() or "Unknown"

def do_file_op(src: Path, dst: Path):
    if DRY_RUN:
        return
    if MODE == "move":
        shutil.move(str(src), str(dst))
    else:
        shutil.copy2(str(src), str(dst))

# 배치 버퍼
buf_imgs = []
buf_rows = []

pbar = tqdm(artworks, desc=f"CLIP Top1 + FILE OPS (bs={BATCH_SIZE}, {MODE})", total=len(artworks))

for row in pbar:
    src = Path(row["image_path"])
    if not src.exists():
        missing += 1
        continue

    img = load_rgb(src)
    if img is None:
        bad += 1
        continue

    buf_imgs.append(img)
    buf_rows.append(row)

    if len(buf_imgs) >= BATCH_SIZE:
        img_in = processor(images=buf_imgs, return_tensors="pt").to(device)

        with torch.no_grad():
            if use_fp16:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    img_feat = model.get_image_features(**img_in)
            else:
                img_feat = model.get_image_features(**img_in)

        img_feat = img_feat / (img_feat.norm(dim=-1, keepdim=True) + 1e-12)
        sim = (img_feat @ text_feat.T).detach().float().cpu().numpy()  # [B, num_labels]

        TOPK = 8  # ✅ genre 후보 최대 8개 (점수 기반 TopK)
        DELTA = 0.015  # ✅ 상대 임계치: score >= top1_score - DELTA (8개 미만 허용)

        top1_idx = np.argmax(sim, axis=1)
        top1_score = sim[np.arange(sim.shape[0]), top1_idx]

        # ✅ TopK(8) 인덱스/점수 추출 (내림차순)
        # num_labels(카테고리 수)가 작아서 argsort로도 충분히 빠름
        topk_idx = np.argsort(sim, axis=1)[:, ::-1][:, :TOPK]
        topk_score = np.take_along_axis(sim, topk_idx, axis=1)

        for rr, i0, sc, k_idx, k_sc in zip(buf_rows, top1_idx, top1_score, topk_idx, topk_score):
            # ✅ 상대 임계치로 TopK 후보 필터링 (최대 8개, 필요시 1개)
            k_idx = np.array(k_idx)
            k_sc  = np.array(k_sc, dtype=np.float32)
            keep = k_sc >= (float(sc) - DELTA)
            k_idx_f = k_idx[keep]
            k_sc_f  = k_sc[keep]
            if k_idx_f.size == 0:
                k_idx_f = k_idx[:1]
                k_sc_f  = k_sc[:1]
            # ✅ primary를 filtered top1으로 강제(불일치 방지)
            i0 = int(k_idx_f[0])
            sc = float(k_sc_f[0])
            # ✅ 상대 임계치로 TopK 후보 필터링 (최대 8개, 필요시 1개)
            k_idx = np.array(k_idx)
            k_sc  = np.array(k_sc, dtype=np.float32)
            keep = k_sc >= (float(sc) - DELTA)
            k_idx_f = k_idx[keep]
            k_sc_f  = k_sc[keep]
            if k_idx_f.size == 0:
                k_idx_f = k_idx[:1]
                k_sc_f  = k_sc[:1]
            # ✅ primary를 filtered top1으로 강제(불일치 방지)
            i0 = int(k_idx_f[0])
            sc = float(k_sc_f[0])
            # ✅ 카테고리 ID (0-based -> 1-based)
            cat_id = int(i0) + 1
            cat_folder = OUT_DIR / f"category{cat_id:03d}"

            if not DRY_RUN:
                cat_folder.mkdir(parents=True, exist_ok=True)

            # ✅ 파일명: category003_0004.ext
            cat_counter[cat_id] += 1
            seq = cat_counter[cat_id]
            src_path = Path(rr["image_path"])
            dst_path = cat_folder / f"category{cat_id:03d}_{seq:04d}{src_path.suffix.lower()}"

            # ✅ 충돌 방지(극히 드물지만 안전)
            if dst_path.exists():
                k = 1
                while (cat_folder / f"category{cat_id:03d}_{seq:04d}__{k}{src_path.suffix.lower()}").exists():
                    k += 1
                dst_path = cat_folder / f"category{cat_id:03d}_{seq:04d}__{k}{src_path.suffix.lower()}"

            try:
                do_file_op(src_path, dst_path)
                placed += 1
            except Exception:
                errors += 1

            # 메타 결과 저장(원하면 나중에 csv/json으로 저장 가능)
            cat_name = idx_to_category(int(i0))
            results.append({
                **rr,
                "primary_genre": cat_name,       # 텍스트 카테고리명
                "primary_score": float(sc),
                "primary_idx0": int(i0),         # 0-based
                "primary_id1": cat_id,           # 1-based (category003용)
                # ✅ TopK 후보(점수 기반) - 후처리에서 genre(top8)로 사용
                "topk_idx0": [int(x) for x in np.array(k_idx_f).tolist()],
                "topk_scores": [float(x) for x in np.array(k_sc_f).tolist()],
                "topk_genres": [idx_to_category(int(x)) for x in np.array(k_idx_f).tolist()],
                "genre": [idx_to_category(int(x)) for x in np.array(k_idx_f).tolist()],  # ✅ 1~8개 가능
                "output_path": str(dst_path),
            })

        # 버퍼 비우기
        buf_imgs, buf_rows = [], []

        # 메모리 정리(선택)
        del img_in, img_feat, sim
        torch.cuda.empty_cache()

# 마지막 찌꺼기 배치 처리
if buf_imgs:
    img_in = processor(images=buf_imgs, return_tensors="pt").to(device)

    with torch.no_grad():
        if use_fp16:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                img_feat = model.get_image_features(**img_in)
        else:
            img_feat = model.get_image_features(**img_in)

    img_feat = img_feat / (img_feat.norm(dim=-1, keepdim=True) + 1e-12)
    sim = (img_feat @ text_feat.T).detach().float().cpu().numpy()

    TOPK = 8  # ✅ genre 후보 최대 8개 (점수 기반 TopK)
    DELTA = 0.015  # ✅ 상대 임계치: score >= top1_score - DELTA (8개 미만 허용)

    top1_idx = np.argmax(sim, axis=1)
    top1_score = sim[np.arange(sim.shape[0]), top1_idx]

    # ✅ TopK(8) 인덱스/점수 추출 (내림차순)
    # num_labels(카테고리 수)가 작아서 argsort로도 충분히 빠름
    topk_idx = np.argsort(sim, axis=1)[:, ::-1][:, :TOPK]
    topk_score = np.take_along_axis(sim, topk_idx, axis=1)

    for rr, i0, sc, k_idx, k_sc in zip(buf_rows, top1_idx, top1_score, topk_idx, topk_score):
        cat_id = int(i0) + 1
        cat_folder = OUT_DIR / f"category{cat_id:03d}"
        if not DRY_RUN:
            cat_folder.mkdir(parents=True, exist_ok=True)

        cat_counter[cat_id] += 1
        seq = cat_counter[cat_id]
        src_path = Path(rr["image_path"])
        dst_path = cat_folder / f"category{cat_id:03d}_{seq:04d}{src_path.suffix.lower()}"

        if dst_path.exists():
            k = 1
            while (cat_folder / f"category{cat_id:03d}_{seq:04d}__{k}{src_path.suffix.lower()}").exists():
                k += 1
            dst_path = cat_folder / f"category{cat_id:03d}_{seq:04d}__{k}{src_path.suffix.lower()}"

        try:
            do_file_op(src_path, dst_path)
            placed += 1
        except Exception:
            errors += 1

        cat_name = idx_to_category(int(i0))
        results.append({
            **rr,
            "primary_genre": cat_name,
            "primary_score": float(sc),
            "primary_idx0": int(i0),
            "primary_id1": cat_id,
            "topk_idx0": [int(x) for x in np.array(k_idx_f).tolist()],
            "topk_scores": [float(x) for x in np.array(k_sc_f).tolist()],
            "topk_genres": [idx_to_category(int(x)) for x in np.array(k_idx_f).tolist()],
                "genre": [idx_to_category(int(x)) for x in np.array(k_idx_f).tolist()],  # ✅ 1~8개 가능
            "output_path": str(dst_path),
        })

    del img_in, img_feat, sim
    torch.cuda.empty_cache()

print("[Done] bad:", bad, "missing:", missing, "placed:", placed, "errors:", errors, "results:", len(results))
print("[OUT_DIR]", OUT_DIR.resolve())
print("[Example]", results[0] if results else None)

[mkdir] /home/j-i14e107/Image_classification/split_category exists?: True


CLIP Top1 + FILE OPS (bs=512, copy): 100%|██████████| 27702/27702 [17:57<00:00, 25.72it/s]  


[Done] bad: 0 missing: 0 placed: 27702 errors: 0 results: 27702
[OUT_DIR] /home/j-i14e107/Image_classification/split_category
[Example] {'artwork_id': 'artwork_000000', 'artist_id': 'train_artwork', 'image_path': 'train_artwork/train_031_022.png', 'primary_genre': 'category096', 'primary_score': 0.28759765625, 'primary_idx0': 95, 'primary_id1': 96, 'topk_idx0': [95, 99, 4, 93, 29], 'topk_scores': [0.28759765625, 0.283935546875, 0.28125, 0.27783203125, 0.274658203125], 'topk_genres': ['category096', 'category100', 'category005', 'category094', 'category030'], 'genre': ['category096', 'category100', 'category005', 'category094', 'category030'], 'output_path': '/home/j-i14e107/Image_classification/split_category/category096/category096_0001.png'}


# 4. 이미지 중복 검증

In [5]:
# Cell 5 : 이미지 검토 및 중복 검증
from pathlib import Path
from collections import Counter

SPLIT_DIR = Path("split_category")
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

if not SPLIT_DIR.exists():
    raise FileNotFoundError(f"폴더가 없습니다: {SPLIT_DIR.resolve()}")

counts = {}
total = 0

for d in sorted([p for p in SPLIT_DIR.iterdir() if p.is_dir()]):
    n = sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS)
    counts[d.name] = n
    total += n

print(f"[Total] {SPLIT_DIR} 이미지 합계: {total}")
print("=== Per-category counts ===")
for k, v in sorted(counts.items(), key=lambda x: (-x[1], x[0])):
    print(f"{k:20s} {v:6d}")

[Total] split_category 이미지 합계: 27702
=== Per-category counts ===
category047            2193
category030            1858
category043            1814
category065            1469
category060            1262
category106            1219
category010             994
category061             923
category039             885
category096             864
category078             719
category020             632
category077             536
category053             527
category056             498
category094             480
category013             459
category031             455
category051             407
category105             379
category006             372
category071             361
category104             306
category107             305
category003             303
category050             286
category022             264
category067             255
category015             245
category103             240
category007             238
category028             237
category027             226
category005

# 5. 작품 성향 계산(클러스터 과정) 및 작가 배치[밑은 잘 됫는지 확인]

In [6]:
# Cell 6: 작가 배치

need = ["results", "text_feat", "ALL_LABELS", "idx_to_category"]
missing = [x for x in need if x not in globals()]
if missing:
    raise RuntimeError(f"필요한 변수가 없어: {missing}\n이전 셀(이미지 스캔/CLIP TopK)을 먼저 실행해줘.")
print("[OK] prerequisites exist:", need)
print("results:", len(results), "text_feat:", tuple(text_feat.shape), "labels:", len(ALL_LABELS))

[OK] prerequisites exist: ['results', 'text_feat', 'ALL_LABELS', 'idx_to_category']
results: 27702 text_feat: (108, 512) labels: 108


# 5.1 카테고리 유사도 측정

In [7]:
# Cell 7 : 카테고리 유사도 측정(108x108 행렬 근사치 생성)

import numpy as np

tf = text_feat.detach().float().cpu().numpy()
tf = tf / (np.linalg.norm(tf, axis=1, keepdims=True) + 1e-12)

cat_sim_text = tf @ tf.T
np.fill_diagonal(cat_sim_text, 1.0)

print("[OK] cat_sim_text shape:", cat_sim_text.shape)

[OK] cat_sim_text shape: (108, 108)


In [8]:
# Cell 8 (FIXED): Category similarity from IMAGE centroids (108x108)
# - 결과(results)에서 top1 카테고리별 centroid 만들고
# - centroid 유사도로 cat_sim_image 계산
# - 이미지 경로는 output_path 우선(이동했을 때 대비)
# - 카테고리 index는 primary_id1/primary_idx0 우선 + 문자열 역매핑 fallback
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path
import re

USE_IMAGE_CENTROIDS = True
MAX_IMAGES_FOR_CENTROID = 200

NUM_CATS = 108  # 고정이면 108, 아니면 text_feat.shape[0] 쓰고 싶으면 바꿔도 됨

if not USE_IMAGE_CENTROIDS:
    cat_sim_image = None
    print("[Skip] image-centroid similarity")
else:
    per_cat_vecs = [[] for _ in range(NUM_CATS)]
    per_cat_count = [0] * NUM_CATS

    # ✅ idx_to_category가 있으면 역매핑 만들어서 "카테고리명"도 idx로 변환 가능하게 함
    name_to_idx0 = {}
    try:
        for i in range(NUM_CATS):
            name_to_idx0[idx_to_category(i)] = i
    except Exception:
        pass

    # 디버그 카운터
    n_no_genre = 0
    n_bad_cat = 0
    n_missing = 0
    n_open_fail = 0
    n_taken = 0

    for r in tqdm(results, desc="Collecting image feats (for centroids)", total=len(results)):
        # --- 1) 카테고리 idx0 구하기 (가장 안전한 순서) ---
        idx0 = None

        if "primary_idx0" in r and r["primary_idx0"] is not None:
            idx0 = int(r["primary_idx0"])
        elif "primary_id1" in r and r["primary_id1"] is not None:
            idx0 = int(r["primary_id1"]) - 1
        else:
            g = r.get("genre")
            if not g:
                n_no_genre += 1
                continue

            top1 = g[0]

            # "category003" / "category003_..." 같은 형태면 숫자만 뽑기
            if isinstance(top1, str):
                m = re.search(r"category\s*0*([0-9]+)", top1, flags=re.IGNORECASE)
                if m:
                    idx0 = int(m.group(1)) - 1
                elif top1 in name_to_idx0:
                    # 카테고리명이 들어있는 경우
                    idx0 = name_to_idx0[top1]

        if idx0 is None or not (0 <= idx0 < NUM_CATS):
            n_bad_cat += 1
            continue

        if per_cat_count[idx0] >= MAX_IMAGES_FOR_CENTROID:
            continue

        # --- 2) 이미지 경로: output_path 우선 (MODE=move에서도 동작) ---
        img_path = Path(r.get("output_path", r.get("image_path", "")))
        if not img_path.exists():
            n_missing += 1
            continue

        # --- 3) 임베딩 추출 ---
        try:
            with Image.open(img_path) as im:
                image = im.convert("RGB")
        except Exception:
            n_open_fail += 1
            continue

        with torch.no_grad():
            img_in = processor(images=image, return_tensors="pt").to(device)
            if use_fp16:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    img_feat = model.get_image_features(**img_in)
            else:
                img_feat = model.get_image_features(**img_in)

        img_feat = img_feat / (img_feat.norm(dim=-1, keepdim=True) + 1e-12)
        v = img_feat.squeeze(0).detach().float().cpu().numpy()

        per_cat_vecs[idx0].append(v)
        per_cat_count[idx0] += 1
        n_taken += 1

    # --- 4) 한 장도 못 모았으면: 그냥 텍스트 기반으로 대체하고 종료 ---
    if max(per_cat_count) == 0:
        cat_sim_image = cat_sim_text.copy()
        print("[WARN] No image feats collected -> fallback to cat_sim_text")
        print(f"debug: taken={n_taken}, no_genre={n_no_genre}, bad_cat={n_bad_cat}, missing={n_missing}, open_fail={n_open_fail}")
    else:
        # centroid 만들기
        first_i = next(i for i, c in enumerate(per_cat_count) if c > 0)
        dim = per_cat_vecs[first_i][0].shape[0]

        centroids = np.zeros((NUM_CATS, dim), dtype=np.float32)
        valid = np.zeros((NUM_CATS,), dtype=bool)

        for i in range(NUM_CATS):
            if per_cat_count[i] == 0:
                continue
            m = np.stack(per_cat_vecs[i], axis=0).mean(axis=0)
            m = m / (np.linalg.norm(m) + 1e-12)
            centroids[i] = m
            valid[i] = True

        # centroid 유사도
        cat_sim_image = centroids @ centroids.T
        np.fill_diagonal(cat_sim_image, 1.0)

        # 유효하지 않은 카테고리는 텍스트 기반으로 대체
        for i in range(NUM_CATS):
            if not valid[i]:
                cat_sim_image[i] = cat_sim_text[i]
                cat_sim_image[:, i] = cat_sim_text[:, i]

        print("[OK] cat_sim_image shape:", cat_sim_image.shape, "| valid centroids:", int(valid.sum()))
        print(f"debug: taken={n_taken}, no_genre={n_no_genre}, bad_cat={n_bad_cat}, missing={n_missing}, open_fail={n_open_fail}")

[OK] cat_sim_image shape: (108, 108) | valid centroids: 107
debug: taken=12059, no_genre=0, bad_cat=0, missing=0, open_fail=0


# 6. 센트로이드 벡터 저장

In [14]:
import json
from pathlib import Path
import numpy as np


OUT_DIR = Path("outputs_json")
OUT_DIR.mkdir(parents=True, exist_ok=True)


OUT_NPY = OUT_DIR / "category_image_centroids.npy" # (NUM_CATS, D)
OUT_META = OUT_DIR / "category_image_centroids_meta.json" # counts/valid/dim 등


# centroids: (NUM_CATS, dim)
# valid: (NUM_CATS,) bool
# per_cat_count: list[int]


np.save(OUT_NPY, centroids.astype(np.float32))


meta = {
    "num_cats": int(NUM_CATS),
    "dim": int(centroids.shape[1]),
    "max_images_per_centroid": int(MAX_IMAGES_FOR_CENTROID),
    "counts": [int(x) for x in per_cat_count],
    "valid": [bool(x) for x in valid],
    "created_at": datetime.now().isoformat(),
}

with open(OUT_META, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)


print("Saved centroids:")
print(" -", OUT_NPY)
print(" -", OUT_META)

NameError: name 'datetime' is not defined

# 7. 텍스트/이미지 유사도 결합

In [9]:
# Cell 9 : 유사도 결합

#ALPAH_TEXT 변수는 실험 변수니 다른 셀에 없다고 착각 ㄴㄴ
ALPHA_TEXT = 0.6  # ✅ 텍스트 비중 (0.6 추천)
if "cat_sim_image" in globals() and cat_sim_image is not None:
    cat_sim = ALPHA_TEXT * cat_sim_text + (1-ALPHA_TEXT) * cat_sim_image
else:
    cat_sim = cat_sim_text

np.fill_diagonal(cat_sim, 1.0)
print("[OK] cat_sim ready:", cat_sim.shape)

[OK] cat_sim ready: (108, 108)


# 8. 카테고리 유사도 기반(의미 인접) 클러스터링/묶음 만들기

In [10]:
# Cell 10 : 클러스트링 빌드(클러스트링 갯수는 CLUSTER_K 변수로 조절하기)

import numpy as np
from sklearn.cluster import KMeans

CLUSTER_K = 31  # ✅ 적절한 분포를 찾아 실험하기.

tf = text_feat.detach().float().cpu().numpy()
tf = tf / (np.linalg.norm(tf, axis=1, keepdims=True) + 1e-12)

kmeans = KMeans(n_clusters=CLUSTER_K, random_state=42, n_init=10)
labels = kmeans.fit_predict(tf)

clusters = []
for c in range(CLUSTER_K):
    members = np.where(labels == c)[0].tolist()
    clusters.append(sorted([int(x) for x in members]))

# 보기 좋게 큰 클러스터부터 정렬(선택)
clusters = sorted(clusters, key=len, reverse=True)

idx_to_cluster = {}
for cid, members in enumerate(clusters):
    for m in members:
        idx_to_cluster[int(m)] = int(cid)

print("[Clusters] count:", len(clusters))
print("[Clusters] sizes:", [len(c) for c in clusters])

[Clusters] count: 31
[Clusters] sizes: [21, 12, 11, 7, 7, 6, 5, 5, 4, 3, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [11]:
# Cell 11 : 각 클러스트의 유사도 측정 및 계산, 배정 후 구체적인 결과 확인.

tf = text_feat.detach().float().cpu().numpy()
tf = tf / (np.linalg.norm(tf, axis=1, keepdims=True) + 1e-12)

cluster_centroids = []
for members in clusters:
    vec = tf[members].mean(axis=0)
    vec = vec / (np.linalg.norm(vec) + 1e-12)
    cluster_centroids.append(vec)
cluster_centroids = np.stack(cluster_centroids, axis=0)

cluster_sim = cluster_centroids @ cluster_centroids.T
np.fill_diagonal(cluster_sim, -1.0)

CLUSTER_ADJ_SIM_MIN = 0.30

def show_cat_neighbors(cat_str="category100", k=12):
    i0 = int(cat_str.replace("category","")) - 1
    nbrs = np.argsort(-cat_sim[i0])[:k]
    print("=== Neighbors of", cat_str, "===")
    for j in nbrs:
        print(f"{idx_to_category(int(j)):<11}  {ALL_LABELS[int(j)]:<28}  sim={cat_sim[i0,int(j)]:.3f}")

show_cat_neighbors("category100", k=12)

=== Neighbors of category100 ===
category100  isometric_illustration        sim=1.000
category077  3d_render_art                 sim=0.904
category094  editorial_illustration        sim=0.902
category101  vector_flat                   sim=0.888
category075  pixel_art                     sim=0.885
category005  animation                     sim=0.884
category076  voxel_art                     sim=0.881
category095  concept_art                   sim=0.866
category079  vr_art                        sim=0.865
category091  photorealism                  sim=0.863
category096  matte_painting                sim=0.863
category073  data_art                      sim=0.862


# 9. 작가 성향 결정

In [12]:
# Cell 12 (REPLACE): results 메타 확인 (절대 results 다시 만들지 말 것)

from collections import Counter

assert "results" in globals(), "results가 없어. Cell 4(Top1 ONLY)를 먼저 실행해."
print("results:", len(results))

cats = [r["primary_genre"] for r in results if r.get("primary_genre")]
print("unique cats:", len(set(cats)))
print("top10:", Counter(cats).most_common(10))

results: 27702
unique cats: 107
top10: [('category047', 2193), ('category030', 1858), ('category043', 1814), ('category065', 1469), ('category060', 1262), ('category106', 1219), ('category010', 994), ('category061', 923), ('category039', 885), ('category096', 864)]


In [18]:
# Cell 13 + 14 (MERGED): Create 5,000 virtual artists + capacities(center=8) + archetypes + needs
# -----------------------------------------------------------------------------
# 목표:
# 1) 작가 수는 무조건 5000명
# 2) 작가 1명당 작품 수(capacity)는 1~20 범위
# 3) capacity 분포는 평균(중심) 8에 가까운 "정규분포 형태"
# 4) 1과 20(가장자리)은 너무 많이 나오지 않게 희소화
# 5) sum(caps) >= num_artworks 를 보장해서 "모든 작품을 중복 없이 배정 가능" 상태 만들기
# 6) archetype 비율대로 작가 성향 생성 + 각 작가별 need(클러스터 요구 시퀀스) 생성
#
# 주의:
# - 이 셀은 파일 복사/이동 같은 작업을 절대 하지 않음.
# - 이 셀의 출력은 virtual_artists (len=5000)와 caps.
# -----------------------------------------------------------------------------

import random
import numpy as np
from collections import Counter

# -----------------------------
# (A) 기본 설정
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

NUM_ARTISTS = 4500           # ✅ 작가 수 고정
MIN_CAP = 1                  # ✅ 작가당 최소 작품 수
MAX_CAP = 20                 # ✅ 작가당 최대 작품 수

# capacity 분포 (정규분포 기반)
CAP_MU = 8.0                 # ✅ 평균 중심(원하는 값)
CAP_SIGMA = 2.4              # ✅ 퍼짐 정도(2.0~3.0 추천, 클수록 다양해짐)
EDGE_ACCEPT_PROB = 0.12      # ✅ 1 또는 20이 샘플링 됐을 때 채택 확률(낮을수록 더 희소)

# 모든 작품을 "중복 없이" 배정하려면 sum(caps) >= num_artworks 필요
ENSURE_ENOUGH_CAPACITY = True

# -----------------------------
# (B) archetype 비율 설정
# -----------------------------
ARCHETYPE_RATIO = {
    "mono_single":       0.18,
    "mono_adjacent":     0.20,
    "hub_spoke":         0.14,
    "balanced_mix":      0.12,
    "dynamic_unrelated": 0.18,
    "complex_oscillate": 0.10,
    "complex_drift":     0.08,
}

# -----------------------------
# (C) need 생성에 사용하는 클러스터 유사도 기준
# -----------------------------
NEAR_SIM = 0.35
FAR_SIM  = 0.15

# prerequisites check (필수 변수 존재 확인)
need_vars = ["results", "clusters", "cluster_sim"]
missing = [x for x in need_vars if x not in globals()]
if missing:
    raise RuntimeError(f"필요한 변수가 없어: {missing}\n이전 셀(results 생성/클러스터링)을 먼저 실행해줘.")

num_artworks = len(results)

# 수용 가능성 체크: 5000명*20=100000이 상한
if ENSURE_ENOUGH_CAPACITY and num_artworks > NUM_ARTISTS * MAX_CAP:
    raise ValueError(
        f"작품({num_artworks})이 너무 많아 5000명*{MAX_CAP}(={NUM_ARTISTS*MAX_CAP})로 수용 불가.\n"
        f"NUM_ARTISTS를 늘리거나 MAX_CAP을 올려야 함."
    )

# -----------------------------
# (D) 클러스터 가중치(클러스터 크기 비례로 선택)
# -----------------------------
cluster_weights = np.array([len(c) for c in clusters], dtype=np.float32)
cluster_weights = cluster_weights / (cluster_weights.sum() + 1e-12)

def pick_cluster_weighted(rng: np.random.Generator):
    return int(rng.choice(len(clusters), p=cluster_weights))

def pick_near_cluster(rng: np.random.Generator, c1: int):
    sims = cluster_sim[c1].copy()
    cand = np.where(sims >= NEAR_SIM)[0]
    cand = cand[cand != c1]
    if len(cand) == 0:
        return int(np.argsort(-sims)[0])  # 가장 비슷한 걸로 fallback
    w = sims[cand]
    w = w - w.min() + 1e-6
    w = w / w.sum()
    return int(rng.choice(cand, p=w))

def pick_far_cluster(rng: np.random.Generator, c1: int):
    sims = cluster_sim[c1].copy()
    cand = np.where(sims <= FAR_SIM)[0]
    cand = cand[cand != c1]
    if len(cand) == 0:
        return int(np.argsort(sims)[0])  # 가장 안비슷한 걸로 fallback
    w = (1 - sims[cand]) * (0.3 + cluster_weights[cand])
    w = w / (w.sum() + 1e-12)
    return int(rng.choice(cand, p=w))

def pick_diverse_clusters(rng: np.random.Generator, k: int):
    picked = [pick_cluster_weighted(rng)]
    while len(picked) < k:
        best, best_score = None, -1e9
        for c in range(len(clusters)):
            if c in picked:
                continue
            sim_avg = float(np.mean([cluster_sim[c, p] for p in picked]))
            score = (1 - sim_avg) + 0.1 * float(cluster_weights[c])
            if score > best_score:
                best_score, best = score, c
        picked.append(int(best))
    return picked

def make_target_sequence(rng: np.random.Generator, archetype: str, capacity: int):
    """
    각 작가가 '어떤 클러스터(성향)를 어느 정도 비율로 원한다(need)'를 만들기 위한 시퀀스 생성기.
    결과 시퀀스를 Counter로 바꾸면 need가 됨.
    """
    if archetype == "mono_single":
        c = pick_cluster_weighted(rng)
        return [c] * capacity

    if archetype == "mono_adjacent":
        c1 = pick_cluster_weighted(rng)
        c2 = pick_near_cluster(rng, c1)
        pool = [c1, c2]
        if rng.random() < 0.35:
            pool.append(pick_near_cluster(rng, c1))
        return list(rng.choice(pool, size=capacity, replace=True))

    if archetype == "dynamic_unrelated":
        pool = pick_diverse_clusters(rng, int(rng.integers(4, 8)))  # 4~7
        return list(rng.choice(pool, size=capacity, replace=True))

    if archetype == "complex_oscillate":
        a = pick_cluster_weighted(rng)
        b = pick_far_cluster(rng, a)
        seq = []
        cur = a
        for _ in range(capacity):
            if rng.random() < 0.45:
                cur = b if cur == a else a
            else:
                cur = pick_near_cluster(rng, cur)
            seq.append(cur)
        return seq

    if archetype == "complex_drift":
        c = pick_cluster_weighted(rng)
        seq = []
        for _ in range(capacity):
            seq.append(c)
            if rng.random() < 0.12:
                c = pick_far_cluster(rng, c)
            else:
                c = pick_near_cluster(rng, c)
        return seq

    if archetype == "hub_spoke":
        hub = pick_cluster_weighted(rng)
        seq = []
        for _ in range(capacity):
            if rng.random() < 0.22:
                seq.append(pick_far_cluster(rng, hub))
            else:
                seq.append(hub)
        return seq

    if archetype == "balanced_mix":
        c1 = pick_cluster_weighted(rng)
        pool = [c1]
        while len(pool) < int(rng.integers(3, 6)):  # 3~5
            sims = cluster_sim[c1].copy()
            cand = np.where((sims > FAR_SIM) & (sims < NEAR_SIM))[0]
            cand = cand[cand != c1]
            if len(cand) == 0:
                cand = np.array([pick_near_cluster(rng, c1), pick_far_cluster(rng, c1)])
            c2 = int(rng.choice(cand))
            if c2 not in pool:
                pool.append(c2)
        return list(rng.choice(pool, size=capacity, replace=True))

    # fallback
    c = pick_cluster_weighted(rng)
    return [c] * capacity


# -----------------------------
# (E) capacity 생성 함수 (5000명, 중심=8, edge 희소, sum(caps) 보정)
# -----------------------------
def make_caps_fixed_centered_normal(
    num_artworks: int,
    num_artists: int,
    min_cap: int,
    max_cap: int,
    mu: float,
    sigma: float,
    accept_edge_prob: float,
    seed: int,
    ensure_capacity_for_all_artworks: bool,
):
    rng = np.random.default_rng(seed)

    # 1) 정규분포 샘플링 + edge 희소화(rejection)
    caps = []
    while len(caps) < num_artists:
        x = int(np.rint(rng.normal(mu, sigma)))
        x = max(min_cap, min(max_cap, x))
        if x in (min_cap, max_cap) and rng.random() > accept_edge_prob:
            continue
        caps.append(x)

    # 2) sum(caps) < num_artworks면 부족분만큼 cap을 올려 수용량 확보
    #    (배정에서 "중복 없이 1작품=1작가"를 유지하고 싶기 때문)
    if ensure_capacity_for_all_artworks:
        diff = num_artworks - sum(caps)
        while diff > 0:
            idxs = [i for i, v in enumerate(caps) if v < max_cap]
            if not idxs:
                break

            # cap이 작은 사람에게 더 자주 +1 하되, 19->20 같은 edge 몰림은 줄임
            weights = []
            for i in idxs:
                v = caps[i]
                w = (max_cap - v)
                if v >= 18:
                    w *= 0.25
                weights.append(w)

            weights = np.array(weights, dtype=np.float64)
            weights = weights / (weights.sum() + 1e-12)
            pick = int(rng.choice(idxs, p=weights))
            caps[pick] += 1
            diff -= 1

    return caps

caps = make_caps_fixed_centered_normal(
    num_artworks=num_artworks,
    num_artists=NUM_ARTISTS,
    min_cap=MIN_CAP,
    max_cap=MAX_CAP,
    mu=CAP_MU,
    sigma=CAP_SIGMA,
    accept_edge_prob=EDGE_ACCEPT_PROB,
    seed=SEED,
    ensure_capacity_for_all_artworks=ENSURE_ENOUGH_CAPACITY,
)

cnt = Counter(caps)
print("[caps] artists:", len(caps),
      "min:", min(caps), "max:", max(caps),
      "avg:", round(sum(caps)/len(caps), 4),
      "sum:", sum(caps),
      "edge(1):", cnt[1], "edge(20):", cnt[20],
      "| num_artworks:", num_artworks)

# -----------------------------
# (F) archetypes 생성 (길이 5000 보장)
# -----------------------------
archetypes = []
for t, r in ARCHETYPE_RATIO.items():
    archetypes += [t] * int(round(NUM_ARTISTS * r))

# 길이 보정(모자라면 mono_adjacent로 채움)
while len(archetypes) < NUM_ARTISTS:
    archetypes.append("mono_adjacent")

# 초과분 자르기 + 섞기
archetypes = archetypes[:NUM_ARTISTS]
random.shuffle(archetypes)

print("[archetypes]", Counter(archetypes))

# -----------------------------
# (G) virtual_artists 최종 생성 (len=5000 확정)
# -----------------------------
rng = np.random.default_rng(SEED)  # need 시퀀스 생성용 RNG

virtual_artists = []
for i, (arch, cap) in enumerate(zip(archetypes, caps)):
    aid = f"v_artist_{i:04d}"
    target_seq = make_target_sequence(rng, arch, int(cap))
    need = Counter(target_seq)

    virtual_artists.append({
        "artist_id": aid,
        "archetype": arch,
        "capacity": int(cap),
        "need": need,
        "assigned": 0,
    })

print("[OK] virtual_artists created:", len(virtual_artists))
print("sample:",
      virtual_artists[0]["artist_id"],
      virtual_artists[0]["archetype"],
      "cap:", virtual_artists[0]["capacity"],
      "need(top5):", virtual_artists[0]["need"].most_common(5))

# (권장) sanity check
assert len(virtual_artists) == NUM_ARTISTS, "virtual_artists length mismatch"
assert len(caps) == NUM_ARTISTS, "caps length mismatch"
assert len(archetypes) == NUM_ARTISTS, "archetypes length mismatch"

[caps] artists: 4500 min: 2 max: 16 avg: 7.9909 sum: 35959 edge(1): 0 edge(20): 0 | num_artworks: 27702
[archetypes] Counter({'mono_adjacent': 900, 'dynamic_unrelated': 810, 'mono_single': 810, 'hub_spoke': 630, 'balanced_mix': 540, 'complex_oscillate': 450, 'complex_drift': 360})
[OK] virtual_artists created: 4500
sample: v_artist_0000 dynamic_unrelated cap: 9 need(top5): [(np.int64(17), 3), (np.int64(3), 3), (np.int64(28), 2), (np.int64(16), 1)]


In [19]:
import copy
virtual_artists_template = copy.deepcopy(virtual_artists)  # ✅ 원본 보관(재실행 안전)

# 10. 작품을 작가에게 배치 후 작가 경향성 확정, json 파일 저장

In [20]:
# 15. 작가의 작품 경향성 정하기.

from collections import defaultdict, deque
from tqdm.auto import tqdm
import random
import copy


# ✅ 템플릿이 아직 없으면 현재 virtual_artists를 템플릿으로 저장
if "virtual_artists_template" not in globals():
    virtual_artists_template = copy.deepcopy(virtual_artists)


# ✅ 항상 템플릿에서 새로 복제해서 사용
virtual_artists = copy.deepcopy(virtual_artists_template)
# 작품별 top1_cluster 계산 (FIXED)
artworks2 = []

bad_cat = 0
bad_cluster = 0

for r in results:
    # ✅ 1) 카테고리 idx0 가져오기: primary_idx0 우선, 없으면 primary_id1 사용
    idx0 = None
    if r.get("primary_idx0") is not None:
        idx0 = int(r["primary_idx0"])
    elif r.get("primary_id1") is not None:
        idx0 = int(r["primary_id1"]) - 1

    if idx0 is None:
        bad_cat += 1
        continue

    # ✅ 2) cluster 매핑
    cid = idx_to_cluster.get(int(idx0), None)
    if cid is None:
        bad_cluster += 1
        continue

    artworks2.append({**r, "top1_cluster": int(cid)})

print("[artworks2]", len(artworks2), "| bad_cat:", bad_cat, "| bad_cluster:", bad_cluster)

cluster_to_q = defaultdict(deque)
for r in artworks2:
    cluster_to_q[int(r["top1_cluster"])].append(r)

def artist_space(a):
    return int(a["capacity"]) - int(a["assigned"])

# --- seed pass0: 각 작가 최소 1개 보장 ---
assigned_rows = []

all_clusters = list(cluster_to_q.keys())

for a in tqdm(virtual_artists, desc="Assign pass0 (seed 1 each)"):
    if artist_space(a) <= 0:
        continue

    # 1) 이 작가가 원하는 need 클러스터 중, 실제 남아있는 작품이 있는 클러스터 우선
    picked_cluster = None
    for c, cnt in a["need"].most_common():
        c = int(c)
        if cnt > 0 and len(cluster_to_q[c]) > 0:
            picked_cluster = c
            break

    # 2) 없으면: 남아있는 작품이 가장 많은 클러스터에서 하나 (빈 큐 제외)
    if picked_cluster is None:
        nonempty = [c for c, q in cluster_to_q.items() if len(q) > 0]
        if not nonempty:
            break
        picked_cluster = max(nonempty, key=lambda x: len(cluster_to_q[int(x)]))
        picked_cluster = int(picked_cluster)

    r = cluster_to_q[picked_cluster].popleft()
    r2 = dict(r)
    r2["artist_id"] = a["artist_id"]
    assigned_rows.append(r2)

    a["assigned"] += 1
    if picked_cluster in a["need"] and a["need"][picked_cluster] > 0:
        a["need"][picked_cluster] -= 1

# --- pass1: quota 채우기 ---
for a in tqdm(virtual_artists, desc="Assign pass1 (quota)"):
    if artist_space(a) <= 0:
        continue
    for c, cnt in list(a["need"].items()):
        c = int(c)
        while cnt > 0 and artist_space(a) > 0 and cluster_to_q[c]:
            r = cluster_to_q[c].popleft()
            r2 = dict(r)
            r2["artist_id"] = a["artist_id"]
            assigned_rows.append(r2)

            a["assigned"] += 1
            cnt -= 1
        a["need"][c] = cnt

# 남은 작품 모으기
remaining = []
for c, q in cluster_to_q.items():
    while q:
        remaining.append(q.popleft())

print("[After pass1] assigned:", len(assigned_rows), "remaining artworks:", len(remaining))

# --- pass2: best-fit로 남은 작품 채우기 ---
def artist_need_centers(a):
    if sum(a["need"].values()) == 0:
        return None
    return [int(c) for c, _ in a["need"].most_common(5)]

def score_artist_for_cluster(a, cid):
    if artist_space(a) <= 0:
        return -1e9
    centers = artist_need_centers(a)
    if centers is None:
        return 0.05 * (artist_space(a)/a["capacity"])
    sim = max([float(cluster_sim[int(cid), int(c)]) for c in centers])
    room = artist_space(a) / a["capacity"]
    jitter = 0.01 * random.random()
    return sim + 0.08 * room + jitter

CANDIDATE_K = 600  # 300~1200 추천. 클수록 정확/느림

for r in tqdm(remaining, desc="Assign pass2 (best-fit)"):
    cid = int(r["top1_cluster"])

    # 아직 자리 있는 작가만 후보로
    candidates = [a for a in virtual_artists if artist_space(a) > 0]
    if not candidates:
        break

    # 너무 많으면 랜덤 샘플
    if len(candidates) > CANDIDATE_K:
        candidates = random.sample(candidates, CANDIDATE_K)

    best_a, best_s = None, -1e9
    for a in candidates:
        s = score_artist_for_cluster(a, cid)
        if s > best_s:
            best_s, best_a = s, a

    if best_a is None:
        continue

    r2 = dict(r)
    r2["artist_id"] = best_a["artist_id"]
    assigned_rows.append(r2)
    best_a["assigned"] += 1

print("[After pass2] assigned:", len(assigned_rows),
      "artists filled:", sum(1 for a in virtual_artists if a["assigned"] >= a["capacity"]))
print("[Check] min assigned:", min(a["assigned"] for a in virtual_artists),
      "max assigned:", max(a["assigned"] for a in virtual_artists))

[artworks2] 27702 | bad_cat: 0 | bad_cluster: 0


Assign pass1 (quota): 100%|██████████| 4500/4500 [00:00<00:00, 129235.78it/s]


[After pass1] assigned: 19234 remaining artworks: 8468


Assign pass2 (best-fit): 100%|██████████| 8468/8468 [00:26<00:00, 322.46it/s]

[After pass2] assigned: 27702 artists filled: 1659
[Check] min assigned: 1 max assigned: 16


In [21]:
# Cell 16 : 구체적으로 정한 경향성 확인하기.
from collections import defaultdict, Counter
import numpy as np

# ===== (선택) 안정성 옵션 =====
MIN_WORKS_PER_ARTIST = 5   # 작품 수가 너무 적은 작가는 통계가 흔들릴 수 있어 필터링(원하면 1로)
WARN_FEW_ARTISTS = 3       # archetype별 작가 수가 이보다 적으면 "참고용" 표시

# 0) (문제4 진단) cluster_sim이 원래 다 높게 나오는지 빠르게 확인
try:
    sim_vals = np.asarray(cluster_sim).ravel()
    print(f"[cluster_sim stats] min={sim_vals.min():.3f}, p50={np.median(sim_vals):.3f}, max={sim_vals.max():.3f}")
except Exception:
    # cluster_sim이 numpy array가 아니면 생략
    pass

by_artist = defaultdict(list)
for r in assigned_rows:
    by_artist[r["artist_id"]].append(r)

def weighted_avg_pairwise_similarity(dist: Counter):
    """
    dist: Counter(cluster_id -> count)
    - 클러스터 종류만 보는 게 아니라, 빈도(count)를 반영해서 평균 유사도를 계산한다.
    - (ci,cj) 쌍의 가중치 = count(ci) * count(cj)
    """
    keys = list(dist.keys())
    if len(keys) <= 1:
        return 1.0

    total_w = 0.0
    total_s = 0.0
    for i in range(len(keys)):
        ci = keys[i]
        for j in range(i+1, len(keys)):
            cj = keys[j]
            w = dist[ci] * dist[cj]
            try:
                s = float(cluster_sim[ci, cj])
            except Exception:
                continue
            total_w += w
            total_s += w * s

    return (total_s / total_w) if total_w > 0 else 1.0

def shannon_entropy(dist: Counter):
    """(선택) 다양성 분포 지표: 분포가 균등할수록 커짐"""
    n = sum(dist.values())
    if n <= 0:
        return 0.0
    ps = np.array([c / n for c in dist.values()], dtype=float)
    return float(-(ps * np.log(ps + 1e-12)).sum())

report = []
for a in virtual_artists:
    aid = a["artist_id"]
    works = by_artist.get(aid, [])
    cids = [w["top1_cluster"] for w in works if w.get("top1_cluster") is not None]
    if not cids:
        continue

    dist = Counter(cids)
    n = len(cids)

    # (선택) 작품 수가 너무 적으면 제외(통계 흔들림 방지)
    if n < MIN_WORKS_PER_ARTIST:
        continue

    dom = dist.most_common(1)[0][1] / n      # 한 클러스터 몰림 정도
    uniq = len(dist)                          # 클러스터 종류 수
    avg_sim = weighted_avg_pairwise_similarity(dist)  # (문제1 개선) 빈도 가중 avg_sim
    ent = shannon_entropy(dist)               # (문제4 보조) 분포 다양성(선택)

    report.append((a["archetype"], dom, uniq, avg_sim, ent, n))

# archetype별 요약
by_type = defaultdict(list)
for t, dom, uniq, avg_sim, ent, n in report:
    by_type[t].append((dom, uniq, avg_sim, ent, n))

print("=== Archetype summary (dom↓, uniq↑, avg_sim↓, entropy↑가 다양성↑) ===")
for t, rows in by_type.items():
    doms = np.array([x[0] for x in rows], dtype=float)
    uniqs = np.array([x[1] for x in rows], dtype=float)
    sims = np.array([x[2] for x in rows], dtype=float)
    ents = np.array([x[3] for x in rows], dtype=float)
    ns = np.array([x[4] for x in rows], dtype=float)

    # (문제3 개선) 작품 수로 가중 평균
    w = ns / max(1.0, ns.sum())

    flag = " (참고용)" if len(rows) < WARN_FEW_ARTISTS else ""
    print(
        f"{t:18s} | artists={len(rows):4d}{flag} "
        f"| dom(wavg)={np.sum(doms*w):.3f} "
        f"| uniq(wavg)={np.sum(uniqs*w):.2f} "
        f"| avg_sim(wavg)={np.sum(sims*w):.3f} "
        f"| entropy(wavg)={np.sum(ents*w):.3f} "
        f"| works(avg)={np.mean(ns):.1f}"
    )

[cluster_sim stats] min=-1.000, p50=0.851, max=0.983
=== Archetype summary (dom↓, uniq↑, avg_sim↓, entropy↑가 다양성↑) ===
dynamic_unrelated  | artists= 462 | dom(wavg)=0.430 | uniq(wavg)=4.05 | avg_sim(wavg)=0.841 | entropy(wavg)=1.252 | works(avg)=6.7
mono_adjacent      | artists= 712 | dom(wavg)=0.574 | uniq(wavg)=3.07 | avg_sim(wavg)=0.934 | entropy(wavg)=0.918 | works(avg)=7.3
mono_single        | artists= 591 | dom(wavg)=0.864 | uniq(wavg)=1.78 | avg_sim(wavg)=0.989 | entropy(wavg)=0.316 | works(avg)=7.9
complex_drift      | artists= 295 | dom(wavg)=0.358 | uniq(wavg)=5.02 | avg_sim(wavg)=0.923 | entropy(wavg)=1.491 | works(avg)=7.0
hub_spoke          | artists= 442 | dom(wavg)=0.849 | uniq(wavg)=1.87 | avg_sim(wavg)=0.987 | entropy(wavg)=0.351 | works(avg)=7.9
complex_oscillate  | artists= 372 | dom(wavg)=0.476 | uniq(wavg)=4.44 | avg_sim(wavg)=0.927 | entropy(wavg)=1.272 | works(avg)=7.4
balanced_mix       | artists= 431 | dom(wavg)=0.472 | uniq(wavg)=3.75 | avg_sim(wavg)=0.925 | e

In [22]:
# Cell 17 : 작가의 작품 및 경향성 정리한 데이터를 json파일로 변환 후 저장하기.

import json
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

OUT_DIR = Path("outputs_json")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def to_py_int(x):
    # np.int64, np.int32 등 -> int
    if isinstance(x, (np.integer,)):
        return int(x)
    return x

def make_jsonable_artist(a: dict) -> dict:
    a2 = dict(a)

    # need: Counter or dict with possibly non-JSON key types
    if "need" in a2 and a2["need"] is not None:
        need = a2["need"]
        if isinstance(need, Counter):
            need_items = need.items()
        elif isinstance(need, dict):
            need_items = need.items()
        else:
            need_items = []

        # JSON에서는 키를 str로 두는 게 안전함
        a2["need"] = {str(to_py_int(k)): int(v) for k, v in need_items}

    # clusters 같은 리스트 안에도 np.int64가 있을 수 있음
    if "clusters" in a2 and isinstance(a2["clusters"], list):
        a2["clusters"] = [to_py_int(x) for x in a2["clusters"]]

    # pref 등도 있을 수 있으니 안전하게 처리
    if "pref" in a2 and isinstance(a2["pref"], list):
        a2["pref"] = [to_py_int(x) for x in a2["pref"]]

    # assigned/capacity 등도 numpy면 int로
    for k in ["assigned", "capacity"]:
        if k in a2:
            a2[k] = int(to_py_int(a2[k]))

    return a2

virtual_artists_jsonable = [make_jsonable_artist(a) for a in virtual_artists]

# 1) 작품 + 장르 + 점수 + artist_id 저장
ARTWORK_ASSIGNED_JSON = OUT_DIR / "artwork_assigned_v2.json"
with ARTWORK_ASSIGNED_JSON.open("w", encoding="utf-8") as f:
    json.dump(assigned_rows, f, ensure_ascii=False, indent=2)

# 2) 작가 저장 (jsonable 버전)
ARTISTS_JSON = OUT_DIR / "artists_v2.json"
with ARTISTS_JSON.open("w", encoding="utf-8") as f:
    json.dump(virtual_artists_jsonable, f, ensure_ascii=False, indent=2)

# 3) 작가 요약
by_artist = defaultdict(list)
for r in assigned_rows:
    by_artist[r["artist_id"]].append(r)

artists_summary = []
for a in virtual_artists_jsonable:
    aid = a["artist_id"]
    works = by_artist.get(aid, [])

    # ✅ top1 카테고리(정합성 우선): primary_genre > genre[0] fallback
    top1_cats = []
    score_by_cat_max = defaultdict(float)
    for w in works:
        cat = w.get("primary_genre")
        if not cat:
            g = w.get("genre")
            if isinstance(g, list) and g:
                cat = g[0]
        if not cat:
            continue
        top1_cats.append(cat)

        # ✅ "가장 높은 점수" 기준: 카테고리별 max(primary_score) 사용
        sc = w.get("primary_score")
        try:
            sc = float(sc) if sc is not None else 0.0
        except Exception:
            sc = 0.0
        if sc > score_by_cat_max[cat]:
            score_by_cat_max[cat] = sc

    c = Counter(top1_cats)

    # ✅ primary_categories: 점수(max) 내림차순 (동점이면 빈도/이름으로 안정화)
    ranked = sorted(score_by_cat_max.items(), key=lambda x: (-x[1], -c.get(x[0], 0), x[0]))
    primary_categories = [k for k, _ in ranked[:5]]

    artists_summary.append({
        "artist_id": aid,
        "archetype": a.get("archetype"),
        "num_works": len(works),
        "primary_categories": primary_categories,
        "top1_distribution": dict(c),
    })

ARTISTS_SUMMARY_JSON = OUT_DIR / "artists_summary_v2.json"
with ARTISTS_SUMMARY_JSON.open("w", encoding="utf-8") as f:
    json.dump(artists_summary, f, ensure_ascii=False, indent=2)

# 4) 유사도/클러스터 저장(있으면)
if "cat_sim" in globals():
    np.save(OUT_DIR / "cat_sim.npy", cat_sim)
if "cluster_sim" in globals():
    np.save(OUT_DIR / "cluster_sim.npy", cluster_sim)
if "clusters" in globals():
    with (OUT_DIR / "clusters.json").open("w", encoding="utf-8") as f:
        json.dump(clusters, f, ensure_ascii=False, indent=2)

print("[Saved]")
print(" -", ARTWORK_ASSIGNED_JSON.resolve())
print(" -", ARTISTS_JSON.resolve())
print(" -", ARTISTS_SUMMARY_JSON.resolve())

[Saved]
 - /home/j-i14e107/Image_classification/outputs_json/artwork_assigned_v2.json
 - /home/j-i14e107/Image_classification/outputs_json/artists_v2.json
 - /home/j-i14e107/Image_classification/outputs_json/artists_summary_v2.json


# 11. 벡터 추출하기.

In [23]:
# Cell: UNIVERSAL batch embedding -> JSONL + JSON(list), works with ANY image folder
# -----------------------------------------------------------------------------
# ✅ 재정립(중요):
# - artist_id: ASSIGN_JSON(artwork_assigned_v2.json)에서만 가져온다. (임의 추정 금지)
# - category/genre: 여기서는 "추정"하지 않는다.
#   * CLIP으로 나온 top1 라벨은 clip_primary_label/clip_primary_score로 저장(옵션)
#   * 다중 라벨(genre 후보)이 ASSIGN_JSON에 이미 있으면 clip_genres로 그대로 저장
# - category라는 필드는 나중에 KMeans cluster_id(20개) 같은 "운영 카테고리"로 쓰는 걸 권장
# -----------------------------------------------------------------------------

import json, contextlib
from pathlib import Path
from collections import Counter
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import open_clip
import numpy as np
from torch.utils.data import Dataset, DataLoader

ImageFile.LOAD_TRUNCATED_IMAGES = True

NOTEBOOK_DIR = Path.cwd().resolve()

# ✅ 배정 JSON
ASSIGN_JSON = NOTEBOOK_DIR / "outputs_json" / "artwork_assigned_v2.json"
if not ASSIGN_JSON.exists():
    ASSIGN_JSON = NOTEBOOK_DIR / "outputs_json" / "artwork_assigned_v2_FIXED.json"
assert ASSIGN_JSON.exists(), f"Assign JSON not found: {ASSIGN_JSON}"

# ✅ 여기만 바꾸면 다른 폴더에서도 그대로 동작
IMAGE_DIR = NOTEBOOK_DIR / "split_category"
assert IMAGE_DIR.exists(), f"IMAGE_DIR not found: {IMAGE_DIR}"

# ✅ outputs
OUT_JSONL = NOTEBOOK_DIR / "outputs_json" / "artwork_vector.jsonl"
OUT_JSON  = NOTEBOOK_DIR / "outputs_json" / "artwork_vector.json"   # list 형태

# -------------------------
# (1) ASSIGN_JSON 로드 + "멀티 키"로 artist_id 매핑 만들기
# -------------------------
with open(ASSIGN_JSON, "r", encoding="utf-8") as f:
    assigned_items = json.load(f)

def norm_slash(s: str) -> str:
    return str(s).replace("\\", "/")

def rel_after_marker(p: str, marker: str):
    s = norm_slash(p)
    m = f"/{marker}/"
    if m in s:
        return s.split(m, 1)[1]
    return None

# key_to_meta: 여러 key(파일명/상대경로/기타)로 artist_id를 찾을 수 있게 구성
key_to_meta = {}              # key -> meta
key_sources = Counter()       # 어떤 키 타입으로 들어갔는지 통계
conflicts = 0

def add_key(k, meta, src):
    global conflicts
    if not k:
        return
    if k in key_to_meta and key_to_meta[k]["artist_id"] != meta["artist_id"]:
        conflicts += 1
        return
    key_to_meta[k] = meta
    key_sources[src] += 1

for it in assigned_items:
    aid = it.get("artist_id")
    if not aid:
        continue

    # ✅ 여기서는 category/genre를 '추정'하지 않음. (있는 것만 그대로 저장)
    meta = {
        "artist_id": aid,
        "clip_primary_label": it.get("primary_genre"),   # CLIP top1 라벨(있으면)
        "clip_primary_score": it.get("primary_score"),   # CLIP top1 점수(있으면)
        "clip_genres": it.get("genre") if isinstance(it.get("genre"), list) else None,  # 이미 있으면 그대로
        "clip_scores": it.get("score") if isinstance(it.get("score"), list) else None, # 이미 있으면 그대로
    }

    # 1) artwork_id가 파일명인 경우가 많음
    if it.get("artwork_id"):
        k = norm_slash(it["artwork_id"])
        add_key(k, meta, "artwork_id")
        add_key(Path(k).name, meta, "artwork_id.basename")

    # 2) split_rel_path가 있으면 상대경로 키로도 추가
    if it.get("split_rel_path"):
        k = norm_slash(it["split_rel_path"])
        add_key(k, meta, "split_rel_path")
        add_key(Path(k).name, meta, "split_rel_path.basename")

    # 3) output_path가 있으면 split_category 기준 상대경로/파일명도 추가
    if it.get("output_path"):
        op = norm_slash(it["output_path"])
        add_key(Path(op).name, meta, "output_path.basename")
        rel = rel_after_marker(op, "split_category")
        if rel:
            add_key(rel, meta, "output_path.after_split_category")
            add_key(Path(rel).name, meta, "output_path.after_split_category.basename")

    # 4) image_path(원본명)도 파일명 키로 추가
    if it.get("image_path"):
        ip = norm_slash(it["image_path"])
        add_key(Path(ip).name, meta, "image_path.basename")

print("[OK] mapping keys:", len(key_to_meta), "| conflicts:", conflicts)
print("[Key sources]", dict(key_sources))

# -------------------------
# (2) IMAGE_DIR 이미지 수집
# -------------------------
IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
all_imgs = [p for p in IMAGE_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
print("[OK] images in IMAGE_DIR:", len(all_imgs))

# -------------------------
# (3) CLIP(openai) 로드
# -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-B-32"
PRETRAINED = "openai"

model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED, device=device)
model.eval()

autocast_ctx = torch.cuda.amp.autocast if device == "cuda" else contextlib.nullcontext
torch.backends.cudnn.benchmark = True
torch.set_grad_enabled(False)

# -------------------------
# (4) 배치 파라미터
# -------------------------
BATCH_SIZE = 1024
NUM_WORKERS = 8
PREFETCH_FACTOR = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True

# -------------------------
# (5) Dataset / DataLoader
# -------------------------
class AnyImageFolderDataset(Dataset):
    def __init__(self, paths, root_dir: Path, key_to_meta: dict):
        self.paths = paths
        self.root_dir = root_dir
        self.key_to_meta = key_to_meta

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        fname = p.name
        rel = None
        try:
            rel = str(p.relative_to(self.root_dir)).replace("\\", "/")
        except Exception:
            rel = None

        # ✅ 멀티키 매칭: rel -> fname 순으로 탐색
        meta = None
        matched_key = None
        if rel and rel in self.key_to_meta:
            meta = self.key_to_meta[rel]
            matched_key = rel
        elif fname in self.key_to_meta:
            meta = self.key_to_meta[fname]
            matched_key = fname

        # ✅ artwork_id는 "이미지 파일명"으로 고정
        artwork_id = fname
        return str(p), artwork_id, rel, meta, matched_key

def collate_fn(batch):
    images, artwork_ids, rel_paths, metas, matched_keys = [], [], [], [], []
    for path_str, artwork_id, rel_path, meta, matched_key in batch:
        if meta is None:
            continue
        try:
            with Image.open(path_str) as im:
                img = im.convert("RGB")
            t = preprocess(img)
        except Exception:
            continue
        images.append(t)
        artwork_ids.append(artwork_id)
        rel_paths.append(rel_path)
        metas.append(meta)
        matched_keys.append(matched_key)

    if not images:
        return None
    return torch.stack(images, 0), artwork_ids, rel_paths, metas, matched_keys

ds = AnyImageFolderDataset(all_imgs, IMAGE_DIR, key_to_meta)
dl = DataLoader(
    ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    persistent_workers=PERSISTENT_WORKERS if NUM_WORKERS > 0 else False,
    collate_fn=collate_fn,
)

# -------------------------
# (6) 배치 임베딩 (512차원 강제)
# -------------------------
def encode_batch(images: torch.Tensor):
    images = images.to(device, non_blocking=True)
    with torch.inference_mode(), autocast_ctx():
        feats = model.encode_image(images)  # [B, D]
        if feats.ndim != 2 or feats.shape[1] != 512:
            raise ValueError(f"CLIP embedding dim must be 512, got shape={tuple(feats.shape)}")
        feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
    return feats

# -------------------------
# (7) 저장(JSONL -> JSON list)
# -------------------------
for p in [OUT_JSONL, OUT_JSON]:
    if p.exists():
        p.unlink()

# 매칭 진단
missing = 0
matched_by = Counter()
for p in all_imgs:
    fname = p.name
    rel = None
    try:
        rel = str(p.relative_to(IMAGE_DIR)).replace("\\", "/")
    except Exception:
        rel = None

    if rel and rel in key_to_meta:
        matched_by["rel"] += 1
    elif fname in key_to_meta:
        matched_by["fname"] += 1
    else:
        missing += 1

print("[Match check] total:", len(all_imgs), "| matched:", len(all_imgs)-missing, "| missing:", missing, "| by:", dict(matched_by))

ok = 0
oom_splits = 0

with open(OUT_JSONL, "w", encoding="utf-8") as f_out:
    pbar = tqdm(dl, desc=f"Embedding batched (bs={BATCH_SIZE})")
    for batch in pbar:
        if batch is None:
            continue
        images, artwork_ids, rel_paths, metas, matched_keys = batch

        try:
            feats = encode_batch(images)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            oom_splits += 1
            if images.shape[0] <= 1:
                raise
            mid = images.shape[0] // 2
            feats = torch.cat([encode_batch(images[:mid]), encode_batch(images[mid:])], 0)

        feats = feats.detach().cpu().float().numpy()  # [B,512]

        for i in range(feats.shape[0]):
            rec = {
                "artist_id": metas[i]["artist_id"],
                "artwork_id": artwork_ids[i],                # ✅ 항상 파일명만
                "image_path": rel_paths[i],              # ✅ IMAGE_DIR 기준 상대경로
                "matched_key": matched_keys[i],              # ✅ 매칭에 사용된 키(디버그)
                # ✅ CLIP 라벨(있으면)
                "clip_primary_label": metas[i].get("clip_primary_label"),
                "clip_primary_score": metas[i].get("clip_primary_score"),
                "clip_genres": metas[i].get("clip_genres"),
                "clip_scores": metas[i].get("clip_scores"),
                # ✅ 임베딩
                "artwork_vector": feats[i].tolist(),
            }
            f_out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            ok += 1

        pbar.set_postfix({"ok": ok, "oom_splits": oom_splits})

print("\n[Step] JSONL saved:", OUT_JSONL, "| records:", ok)

vec_list = []
with open(OUT_JSONL, "r", encoding="utf-8") as f_in:
    for line in tqdm(f_in, desc="Convert JSONL -> JSON(list)"):
        vec_list.append(json.loads(line))

with open(OUT_JSON, "w", encoding="utf-8") as f_out:
    json.dump(vec_list, f_out, ensure_ascii=False, indent=2)

print("\n✅ Done")
print(" - IMAGE_DIR:", IMAGE_DIR)
print(" - ok:", ok)
print(" - oom_splits:", oom_splits)
print(" - saved JSONL:", OUT_JSONL)
print(" - saved JSON(list):", OUT_JSON)


[OK] mapping keys: 110808 | conflicts: 0
[Key sources] {'artwork_id': 27702, 'artwork_id.basename': 27702, 'output_path.basename': 27702, 'output_path.after_split_category': 27702, 'output_path.after_split_category.basename': 27702, 'image_path.basename': 27702}
[OK] images in IMAGE_DIR: 27702


/home/j-i14e107/.conda/envs/ai_dev_env/lib/python3.9/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


[Match check] total: 27702 | matched: 27702 | missing: 0 | by: {'rel': 27702}


/tmp/ipykernel_793376/181002981.py:217: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), autocast_ctx():
Embedding batched (bs=1024): 100%|██████████| 28/28 [04:21<00:00,  9.33s/it, ok=27702, oom_splits=0] 



[Step] JSONL saved: /home/j-i14e107/Image_classification/outputs_json/artwork_vector.jsonl | records: 27702


Convert JSONL -> JSON(list): 27702it [00:04, 6032.89it/s]



✅ Done
 - IMAGE_DIR: /home/j-i14e107/Image_classification/split_category
 - ok: 27702
 - oom_splits: 0
 - saved JSONL: /home/j-i14e107/Image_classification/outputs_json/artwork_vector.jsonl
 - saved JSON(list): /home/j-i14e107/Image_classification/outputs_json/artwork_vector.json


In [24]:
# Postprocess: artwork_assigned_v2.json만 "genre(top8)" 규격으로 정리
# - genre: 최대 8개 (점수 높은 순). 8개 미만이면 있는 만큼만.
# - score 정보는 저장하지 않음(genre_scores/primary_score/topk_scores 등 제거)
#
# ✅ artist_id / primary_genre 등 '원본 필드'는 그대로 두고,
#    genre 관련 필드만 "있는 것만" 정리해서 덮어쓴다.

import json
from pathlib import Path
import time

NOTEBOOK_DIR = Path.cwd().resolve()
ASSIGN_PATH  = NOTEBOOK_DIR / "outputs_json" / "artwork_assigned_v2.json"

assert ASSIGN_PATH.exists(), f"Not found: {ASSIGN_PATH}"

TOPK = 8
DELTA = 0.015  # 상대 임계치: score >= top1_score - DELTA

def _as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def extract_genre_topk(it: dict, topk: int = 8):
    """가능한 모든 케이스를 포괄해서 (label, score) 후보를 모은 뒤,
    score 내림차순으로 정렬해 label만 topk개 반환.
    score 정보가 없으면, 이미 들어있는 순서를 그대로 사용.
    """
    cand = []

    # 1) (labels + scores) 형태
    # - 예: {"genres":[...], "scores":[...]} 또는 {"genre":[...], "genre_scores":[...]}
    for lab_key, sc_key in [
        ("genres", "scores"),
        ("genre", "genre_scores"),
        ("topk_genres", "topk_scores"),
        ("topk_labels", "topk_scores"),
        ("labels", "scores"),
    ]:
        if lab_key in it and sc_key in it:
            labs = _as_list(it.get(lab_key))
            scs  = _as_list(it.get(sc_key))
            if len(labs) == len(scs) and len(labs) > 0:
                for l, s in zip(labs, scs):
                    try:
                        cand.append((str(l), float(s)))
                    except Exception:
                        pass

    # 2) dict 형태: {"label": score, ...}
    for dict_key in ["score_map", "genre_score_map", "topk_score_map"]:
        if dict_key in it and isinstance(it[dict_key], dict):
            for l, s in it[dict_key].items():
                try:
                    cand.append((str(l), float(s)))
                except Exception:
                    pass

    # 3) topk_idx0/topk_scores -> category### 로 변환 가능한 경우
    if "topk_idx0" in it and "topk_scores" in it:
        idxs = _as_list(it.get("topk_idx0"))
        scs  = _as_list(it.get("topk_scores"))
        if len(idxs) == len(scs) and len(idxs) > 0:
            for i0, s in zip(idxs, scs):
                try:
                    i0 = int(i0)
                    lab = f"category{i0+1:03d}"
                    cand.append((lab, float(s)))
                except Exception:
                    pass
    # 4) topk_genres(list)만 있는 케이스 (점수는 없지만 순서는 이미 topK 순이라고 가정)
    if not cand:
        if isinstance(it.get("topk_genres"), list) and it["topk_genres"]:
            labs = [str(x) for x in it["topk_genres"]]
            seen=set(); out=[]
            for l in labs:
                if l not in seen:
                    out.append(l); seen.add(l)
                if len(out) >= topk:
                    break
            return out

    # 5) primary_genre만 있는 케이스 (topk 후보가 없으면 최소한 이거라도)
    if not cand:
        if it.get("primary_genre") is not None:
            return [str(it["primary_genre"])]
        # 이미 genre(list)만 있고 점수는 없는 경우: 그대로 topk로 자르기
        if isinstance(it.get("genre"), list) and it["genre"]:
            labs = [str(x) for x in it["genre"]]
            # 중복 제거(순서 유지)
            seen=set(); out=[]
            for l in labs:
                if l not in seen:
                    out.append(l); seen.add(l)
                if len(out) >= topk:
                    break
            return out
        return []

    # 후보가 여러 소스에서 중복으로 들어올 수 있으니, 같은 label은 최고 점수로 합치기
    best = {}
    for l, s in cand:
        if l not in best or s > best[l]:
            best[l] = s

    # 점수 내림차순 정렬
    ranked = sorted(best.items(), key=lambda x: x[1], reverse=True)
    labels = [l for l, _ in ranked[:topk]]
    return labels

# backup
ts = time.strftime("%Y%m%d_%H%M%S")
bak = ASSIGN_PATH.with_suffix(f".json.bak_{ts}")
bak.write_text(ASSIGN_PATH.read_text(encoding="utf-8"), encoding="utf-8")
print("[Backup]", bak)

assigned = json.loads(ASSIGN_PATH.read_text(encoding="utf-8"))

drop_keys = {
    "primary_score",
    "scores",
    "genre_scores",
    "topk_scores",
    "score_map",
    "genre_score_map",
    "topk_score_map",
}

changed = 0
for it in assigned:
    new_genre = extract_genre_topk(it, topk=TOPK)

    # ✅ 상대 임계치 적용 (가능한 경우): top1(primary_score) 대비 DELTA 이내만 남김
    if isinstance(it.get("topk_genres"), list) and isinstance(it.get("topk_scores"), list) and it.get("primary_score") is not None:
        try:
            psc = float(it["primary_score"])
            labs = [str(x) for x in it["topk_genres"]]
            scs  = [float(x) for x in it["topk_scores"]]
            if len(labs) == len(scs) and len(labs) > 0:
                filtered = [l for l, s in zip(labs, scs) if s >= (psc - DELTA)]
                if filtered:
                    new_genre = filtered[:TOPK]
                else:
                    new_genre = [labs[0]]
        except Exception:
            pass

    # ✅ 가능하면 primary_genre와 genre[0] 일치시키기:
    # - output_path에 있는 category### 폴더가 있으면 그 값을 primary로 우선(파일 위치와 정합)
    import re as _re
    op = it.get("output_path") or ""
    m = _re.search(r"(category\d{3})", op)
    if m:
        primary_from_path = m.group(1)
        it["primary_genre"] = primary_from_path
        if new_genre:
            if primary_from_path in new_genre:
                new_genre = [primary_from_path] + [g for g in new_genre if g != primary_from_path]
            else:
                new_genre = [primary_from_path] + new_genre
                new_genre = new_genre[:TOPK]

    if new_genre:
        it["genre"] = new_genre
        changed += 1
    else:
        it["genre"] = []
        
    # ------------------------------------------------------------
    # (B) topk_* 를 최종 genre 와 완전히 동기화 + primary 인덱스도 맞추기
    # ------------------------------------------------------------
    def _cat_to_idx0(cat: str) -> int:
        # "category001" -> 0
        return int(cat.replace("category", "")) - 1

    # primary_genre / primary_idx0 / primary_id1 정합성 강제 (genre[0] 기준)
    if isinstance(it.get("genre"), list) and it["genre"]:
        it["primary_genre"] = it["genre"][0]
        it["primary_idx0"] = _cat_to_idx0(it["primary_genre"])
        it["primary_id1"]  = it["primary_idx0"] + 1

        # (B) 핵심: topk_*를 genre와 동일하게
        it["topk_genres"] = list(it["genre"])
        it["topk_idx0"]   = [_cat_to_idx0(g) for g in it["genre"]]
        
    # score 관련 키 제거 (요청사항)
    for k in list(it.keys()):
        if k in drop_keys:
            it.pop(k, None)

ASSIGN_PATH.write_text(json.dumps(assigned, ensure_ascii=False, indent=2), encoding="utf-8")

# ------------------------------------------------------------
# (추가) artists_summary_v2.json도 함께 재생성 (primary_categories 유지)
# ------------------------------------------------------------
from collections import defaultdict, Counter

SUMMARY_PATH = NOTEBOOK_DIR / "outputs_json" / "artists_summary_v2.json"
by_artist = defaultdict(list)
for r in assigned:
    aid = r.get("artist_id")
    if aid is None:
        continue
    by_artist[aid].append(r)

artists_summary = []
for aid, works in by_artist.items():
    # top1 기준: primary_genre 우선, 없으면 genre[0]
    top1 = []
    for w in works:
        pg = w.get("primary_genre")
        if pg is None:
            pg = w.get("clip_primary_label")  # alt schema
        if pg is None and isinstance(w.get("genre"), list) and w["genre"]:
            pg = w["genre"][0]
        if pg is None and isinstance(w.get("topk_genres"), list) and w["topk_genres"]:
            pg = w["topk_genres"][0]
        if pg is None and isinstance(w.get("clip_genres"), list) and w["clip_genres"]:
            pg = w["clip_genres"][0]
        if pg is not None:
            top1.append(str(pg))
    c = Counter(top1)
    artists_summary.append({
        "artist_id": aid,
        "num_works": len(works),
        "primary_categories": [k for k, _ in c.most_common(5)],
        "top1_distribution": dict(c),
    })

# 작품 많은 순으로 정렬(보기 편하게)
artists_summary.sort(key=lambda x: x.get("num_works", 0), reverse=True)

SUMMARY_PATH.write_text(json.dumps(artists_summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("[OK] regenerated:", SUMMARY_PATH)
print(f"[OK] updated: {ASSIGN_PATH}")
print(" - rows:", len(assigned))
print(" - genre updated:", changed)
print(" - TOPK:", TOPK)


[Backup] /home/j-i14e107/Image_classification/outputs_json/artwork_assigned_v2.json.bak_20260127_154635
[OK] regenerated: /home/j-i14e107/Image_classification/outputs_json/artists_summary_v2.json
[OK] updated: /home/j-i14e107/Image_classification/outputs_json/artwork_assigned_v2.json
 - rows: 27702
 - genre updated: 27702
 - TOPK: 8


In [30]:
# Diagnostics: assigned/summary integrity checks
import json
from pathlib import Path

p_assign = Path('outputs_json')/'artwork_assigned_v2.json'
p_sum = Path('outputs_json')/'artists_summary_v2.json'
assigned = json.loads(p_assign.read_text(encoding='utf-8'))
print('assigned rows:', len(assigned))
empty_genre = sum(1 for r in assigned if not (isinstance(r.get('genre'), list) and len(r['genre'])>0))
missing_artist = sum(1 for r in assigned if r.get('artist_id') is None)
print('empty genre rows:', empty_genre)
print('missing artist_id rows:', missing_artist)

# duplicate artwork_id check
ids=[r.get('artwork_id') for r in assigned]
dups=len(ids)-len(set(ids))
print('duplicate artwork_id:', dups)

if p_sum.exists():
    summ = json.loads(p_sum.read_text(encoding='utf-8'))
    print('artists_summary rows:', len(summ))
    empty_pc = sum(1 for a in summ if not a.get('primary_categories'))
    print('artists with empty primary_categories:', empty_pc)
    if len(summ)>0:
        print('sample:', summ[0])


assigned rows: 1000
empty genre rows: 0
missing artist_id rows: 0
duplicate artwork_id: 0
artists_summary rows: 80
artists with empty primary_categories: 0
sample: {'artist_id': 'v_artist_0011', 'num_works': 17, 'primary_categories': ['category039', 'category086', 'category041', 'category094', 'category050'], 'top1_distribution': {'category050': 1, 'category031': 1, 'category003': 1, 'category039': 2, 'category046': 1, 'category086': 2, 'category011': 1, 'category020': 1, 'category028': 1, 'category041': 2, 'category094': 2, 'category082': 1, 'category013': 1}}


In [25]:
import json
from pathlib import Path

# 입력/출력 경로
IN_PATH  = Path("outputs_json/artwork_assigned_v2.json")   # 필요시 경로 수정
OUT_PATH = IN_PATH.with_name(IN_PATH.stem + "_post" + IN_PATH.suffix)

assert IN_PATH.exists(), f"Not found: {IN_PATH.resolve()}"

def extract_stem_from_output_path(p):
    if not p:
        return None
    try:
        return Path(p).stem  # category094_0001.png -> category094_0001
    except Exception:
        return None

with IN_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

# data가 list / dict 둘 다 대응
items = data if isinstance(data, list) else data.get("items", None)
if items is None:
    raise ValueError("JSON 구조가 list도 아니고 {'items': [...]} 형태도 아닙니다. 파일 구조를 확인해주세요.")

changed = 0
skipped = 0

for it in items:
    outp = it.get("output_path") or it.get("outputPath")
    stem = extract_stem_from_output_path(outp)
    if stem is None:
        skipped += 1
        continue

    it["artwork_id"] = stem
    changed += 1

with OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"[DONE] saved: {OUT_PATH}")
print(f"changed={changed}, skipped={skipped}")

[DONE] saved: outputs_json/artwork_assigned_v2_post.json
changed=27702, skipped=0


In [26]:
from pathlib import Path
import json
import random
import numpy as np

BASE_DIR = Path.cwd()
ASSIGNED_POST = BASE_DIR / "outputs_json" / "artwork_assigned_v2_post.json"
VEC_PATH = Path.cwd() / "outputs_json" / "artwork_vector.json"  # 또는 post 파일
vec_raw = json.load(open(VEC_PATH, "r", encoding="utf-8"))

def build_vec_dict(vec_raw):
    # dict 포맷: { "category000_0000.png": [vec], ... }
    if isinstance(vec_raw, dict):
        return {Path(str(k)).stem: v for k, v in vec_raw.items()}

    # list 포맷: [{"artwork_id":"...","artwork_vector":[...]}, ...]
    out = {}
    for r in vec_raw:
        if not isinstance(r, dict): 
            continue
        vid = r.get("artwork_id") or r.get("item_id") or r.get("id")
        vec = r.get("artwork_vector") or r.get("vector") or r.get("embedding")
        if vid is None or vec is None:
            continue
        out[Path(str(vid)).stem] = vec
    return out

artwork_vector = build_vec_dict(vec_raw)
AVAILABLE_IDS = list(artwork_vector.keys())

print("vector keys:", len(AVAILABLE_IDS), "sample:", AVAILABLE_IDS[:10])

def load_json(p: Path):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

from pathlib import Path

def parse_vector_json(vec_data):
    """
    returns vec_dict: {artwork_id_stem(str): vector(list[float])}
    supports:
      - dict: {id: [vec]}
      - list: [{"artwork_id":..., "artwork_vector":[...]}] (or item_id/vector/embedding)
    """
    vec_dict = {}

    def _stem(x):
        return Path(str(x)).stem  # '.../a.png' -> 'a'

    if isinstance(vec_data, dict):
        for k, v in vec_data.items():
            if isinstance(v, list):
                vec_dict[_stem(k)] = v
        return vec_dict

    if isinstance(vec_data, list):
        for r in vec_data:
            if not isinstance(r, dict):
                continue
            vid = r.get("artwork_id") or r.get("item_id") or r.get("id")
            vec = r.get("artwork_vector") or r.get("vector") or r.get("embedding")
            if vid is None or vec is None:
                continue
            vec_dict[_stem(vid)] = vec
        return vec_dict

    raise ValueError("Unsupported artwork_vector.json format")

vec_raw  = load_json(VEC_PATH)
vec_dict = parse_vector_json(vec_raw)   # ✅ 이제 stem 키로 통일됨

assigned = load_json(ASSIGNED_POST)

assigned_ids = [str(r["artwork_id"]) for r in assigned if isinstance(r, dict) and r.get("artwork_id") is not None]
assigned_set = set(assigned_ids)
vec_set = set(vec_dict.keys())

inter = assigned_set & vec_set
missing = list(assigned_set - vec_set)

print("assigned unique:", len(assigned_set))
print("vector unique  :", len(vec_set))
print("matched        :", len(inter))
print("missing        :", len(missing))
print("match rate     :", (len(inter) / max(len(assigned_set), 1)) * 100, "%")

# 벡터 차원 체크 (샘플 10개)
sample_ids = random.sample(list(inter), k=min(10, len(inter))) if inter else []
dims = []
bad = []
for aid in sample_ids:
    v = vec_dict[aid]
    if not isinstance(v, list):
        bad.append(aid)
        continue
    dims.append(len(v))

print("\nvector dim sample:", dims[:10])
if dims:
    print("dim min/max:", min(dims), max(dims))
if bad:
    print("non-list vector sample ids:", bad[:5])

# 누락 샘플 출력
print("\nmissing sample:", missing[:20])

# backward-compatible alias
AVAILABLE_ARTWORK_IDS = AVAILABLE_IDS


vector keys: 27702 sample: ['category030_1137', 'category030_0095', 'category030_1534', 'category030_1208', 'category030_1118', 'category030_1372', 'category030_0315', 'category030_1322', 'category030_0305', 'category030_0789']
assigned unique: 27702
vector unique  : 27702
matched        : 27702
missing        : 0
match rate     : 100.0 %

vector dim sample: [512, 512, 512, 512, 512, 512, 512, 512, 512, 512]
dim min/max: 512 512

missing sample: []


In [27]:
import json
import re
import numpy as np
from pathlib import Path
from datetime import datetime

NOTEBOOK_DIR = Path.cwd().resolve()
OUT_DIR = NOTEBOOK_DIR / "outputs_json"
VEC_PATH = OUT_DIR / "artwork_vector.json"   # list or dict 가능

TOPK = 200
MIN_K = 20
EPS = 1e-12

OUT_CENTROIDS_JSON = OUT_DIR / "category_centroids_topk.json"
OUT_CENTROIDS_META = OUT_DIR / "category_centroids_topk_meta.json"

assert VEC_PATH.exists(), f"Not found: {VEC_PATH}"

def l2norm(x: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / (n + EPS)

# category030_0006.png -> "category030"
CAT_RE = re.compile(r"^(category\d{3})_\d{4}\.(?:png|jpg|jpeg|webp|bmp)$", re.IGNORECASE)

def load_vectors_to_pairs(path: Path):
    """return list of (key_str, vec_np)"""
    raw = json.load(open(path, "r", encoding="utf-8"))
    pairs = []
    dim = None

    if isinstance(raw, dict):
        for k, v in raw.items():
            vec = np.asarray(v, dtype=np.float32).reshape(-1)
            if dim is None: dim = int(vec.shape[0])
            if vec.shape[0] != dim: 
                continue
            pairs.append((str(k), vec))
        return pairs, dim, {"mode": "dict"}

    if isinstance(raw, list):
        for r in raw:
            if not isinstance(r, dict): 
                continue
            k = r.get("artwork_id") or r.get("id") or r.get("item_id")
            v = r.get("artwork_vector") or r.get("vector") or r.get("embedding")
            if k is None or v is None:
                continue
            vec = np.asarray(v, dtype=np.float32).reshape(-1)
            if dim is None: dim = int(vec.shape[0])
            if vec.shape[0] != dim:
                continue
            pairs.append((str(k), vec))
        return pairs, dim, {"mode": "list"}

    raise ValueError("Unsupported vector json format")

pairs, dim, meta0 = load_vectors_to_pairs(VEC_PATH)
print(f"[OK] loaded vectors: {len(pairs):,} | dim={dim} | mode={meta0['mode']}")

# 카테고리별 수집
per_cat = {}
bad_key = 0
for k, vec in pairs:
    name = Path(k).name  # 혹시 경로여도 파일명만
    m = CAT_RE.match(name)
    if not m:
        bad_key += 1
        continue
    cat = m.group(1).lower()  # "category030"
    per_cat.setdefault(cat, []).append(vec)

print(f"[OK] parsed categories: {len(per_cat):,} | bad_key={bad_key:,}")

# Top-K centroid
centroid_records = []
counts = {}
used_topk = {}

for cat, vecs in per_cat.items():
    X = np.stack(vecs, axis=0).astype(np.float32)
    X = l2norm(X)
    n = X.shape[0]

    if n < max(MIN_K, 2):
        c = X.mean(axis=0)
        c = c / (np.linalg.norm(c) + EPS)
        k_used = n
    else:
        c0 = X.mean(axis=0)
        c0 = c0 / (np.linalg.norm(c0) + EPS)
        sims = X @ c0
        k = min(TOPK, n)
        top_idx = np.argsort(-sims)[:k]
        c = X[top_idx].mean(axis=0)
        c = c / (np.linalg.norm(c) + EPS)
        k_used = k

    centroid_records.append({
        "category": cat,                 # ✅ "category030"
        "centroid": c.astype(float).tolist()  # ✅ 512차원 리스트
    })
    counts[cat] = int(n)
    used_topk[cat] = int(k_used)

OUT_DIR.mkdir(parents=True, exist_ok=True)
json.dump(centroid_records, open(OUT_CENTROIDS_JSON, "w", encoding="utf-8"), ensure_ascii=False)

meta = {
    "created_at": datetime.now().isoformat(),
    "vector_dim": int(dim),
    "num_categories": int(len(centroid_records)),
    "topk": int(TOPK),
    "min_k_fallback_mean": int(MIN_K),
    "counts_per_category": counts,
    "used_k_per_category": used_topk,
    "bad_key_rows": int(bad_key),
    "vector_file": str(VEC_PATH),
    "vector_file_mode": meta0["mode"],
    "output_json": str(OUT_CENTROIDS_JSON),
}
json.dump(meta, open(OUT_CENTROIDS_META, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

print(f"[SAVE] {OUT_CENTROIDS_JSON} | categories={len(centroid_records)} | dim={dim}")
print(f"[SAVE] {OUT_CENTROIDS_META}")

[OK] loaded vectors: 27,702 | dim=512 | mode=list
[OK] parsed categories: 107 | bad_key=0
[SAVE] /home/j-i14e107/Image_classification/outputs_json/category_centroids_topk.json | categories=107 | dim=512
[SAVE] /home/j-i14e107/Image_classification/outputs_json/category_centroids_topk_meta.json


In [28]:
import json
import re
import os
from pathlib import Path

# ==========================================
# 설정
# ==========================================
TARGET_FILE = Path("outputs_json/artwork_vector.json")

def clean_id(raw_id):
    if not raw_id: return raw_id
    
    # 1. 경로가 포함되어 있다면 제거 (예: a/b/c.jpg -> c.jpg)
    filename = os.path.basename(str(raw_id))
    
    # 2. 정규표현식으로 확장자 강제 제거 (대소문자 무관)
    # 예: .jpg, .JPG, .png, .webp 등을 찾아서 빈 문자열로 바꿈
    new_id = re.sub(r'\.(jpg|jpeg|png|webp|bmp|gif|tiff)$', '', filename, flags=re.IGNORECASE)
    
    return new_id

def main():
    print(f"📂 현재 폴더: {Path.cwd()}")
    
    if not TARGET_FILE.exists():
        print(f"❌ 오류: '{TARGET_FILE}' 파일이 없습니다. 파일명을 확인해주세요.")
        return

    print(f"📖 파일 로딩 중: {TARGET_FILE}")
    with open(TARGET_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    modified_count = 0
    sample_printed = 0
    
    print("\n🔍 변경 내용 미리보기 (상위 5개):")
    print("-" * 50)

    for item in data:
        # artwork_id가 없으면 다른 후보 키 확인 (id, piece_id 등)
        original_id = item.get("artwork_id") or item.get("id")
        
        if original_id:
            new_id = clean_id(original_id)
            
            # 변경사항이 있으면 적용
            if original_id != new_id:
                # 데이터 업데이트
                item["artwork_id"] = new_id
                
                # 로그 출력 (최대 5개까지만)
                if sample_printed < 5:
                    print(f"🛠️ 수정: '{original_id}' -> '{new_id}'")
                    sample_printed += 1
                
                modified_count += 1

    print("-" * 50)
    
    if modified_count == 0:
        print("✅ 변경할 내용이 없습니다! (이미 확장자가 없거나 ID가 깨끗합니다)")
    else:
        print(f"🚀 총 {modified_count}개의 ID를 수정했습니다.")
        
        # 저장
        print(f"💾 저장 중...")
        with open(TARGET_FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print("🎉 저장 완료!")

if __name__ == "__main__":
    main()

📂 현재 폴더: /home/j-i14e107/Image_classification
📖 파일 로딩 중: outputs_json/artwork_vector.json

🔍 변경 내용 미리보기 (상위 5개):
--------------------------------------------------
🛠️ 수정: 'category030_1137.jpg' -> 'category030_1137'
🛠️ 수정: 'category030_0095.jpg' -> 'category030_0095'
🛠️ 수정: 'category030_1534.jpg' -> 'category030_1534'
🛠️ 수정: 'category030_1208.jpg' -> 'category030_1208'
🛠️ 수정: 'category030_1118.jpg' -> 'category030_1118'
--------------------------------------------------
🚀 총 27702개의 ID를 수정했습니다.
💾 저장 중...
🎉 저장 완료!


In [29]:
import json
import numpy as np
from pathlib import Path
from datetime import datetime

# ==========================================
# 1. 설정
# ==========================================
NOTEBOOK_DIR = Path.cwd().resolve()
OUT_DIR = NOTEBOOK_DIR / "outputs_json"
ASSIGN_JSON = OUT_DIR / "artwork_assigned_v2_post.json"
VEC_PATH    = OUT_DIR / "artwork_vector.json"

TOPK = 200
MIN_K = 20
EPS = 1e-12

OUT_CENTROIDS_JSON = OUT_DIR / "category_centroids_topk.json"
OUT_CENTROIDS_META = OUT_DIR / "category_centroids_topk_meta.json"

assert ASSIGN_JSON.exists(), f"Not found: {ASSIGN_JSON}"
assert VEC_PATH.exists(), f"Not found: {VEC_PATH}"

# ==========================================
# 2. 유틸 함수
# ==========================================
def l2norm(x: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / (n + EPS)

def normalize_id(raw_id):
    if raw_id is None:
        return None
    return Path(str(raw_id)).stem

def pick_first(d: dict, keys):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None

def extract_artwork_id(it: dict):
    v = pick_first(it, ["artwork_id", "artworkId", "id", "piece_id", "stem"])
    if v is not None:
        return normalize_id(v)
    
    p = pick_first(it, ["image_path", "path", "filename"])
    if p:
        return normalize_id(p)
    return None

def extract_category(it: dict):
    # [수정] "primary_genre", "genre" 키 추가!
    v = pick_first(it, [
        "category", "primary_category", "label", 
        "primary_genre", "genre", "primary_label"
    ])
    if v is None:
        return None
    # 리스트인 경우 첫 번째 요소 사용 (예: ["cat1", "cat2"])
    if isinstance(v, list):
        if len(v) > 0:
            return str(v[0])
        return None
    return str(v)

# ==========================================
# 3. 벡터 로드
# ==========================================
def load_vectors_any(path: Path):
    vec_map = {}
    dim = None
    
    with open(path, "r", encoding="utf-8") as f:
        first_char = f.read(1)
    
    mode = "json_array" if first_char == "[" else "jsonl"
    
    with open(path, "r", encoding="utf-8") as f:
        if mode == "json_array":
            try:
                iterable = json.load(f)
            except:
                iterable = []
        else:
            iterable = f

        for item in iterable:
            if mode == "jsonl":
                if isinstance(item, str):
                    try: rec = json.loads(item)
                    except: continue
                else: rec = item
            else:
                rec = item

            aid = rec.get("artwork_id") or rec.get("id")
            v = rec.get("artwork_vector") or rec.get("vector")
            
            if aid is None or v is None:
                continue
                
            clean_id = normalize_id(aid)
            v_arr = np.asarray(v, dtype=np.float32).reshape(-1)
            if dim is None: dim = v_arr.shape[0]
            
            if v_arr.shape[0] == dim:
                vec_map[clean_id] = v_arr

    return vec_map, dim

print(">>> Loading Vectors...")
vec_map, dim = load_vectors_any(VEC_PATH)
print(f"[OK] Vectors loaded: {len(vec_map):,} (dim={dim})")
print(f"   (Sample Vector IDs: {list(vec_map.keys())[:3]})")

# ==========================================
# 4. 할당 정보 로드
# ==========================================
print(">>> Loading Assignments...")
with open(ASSIGN_JSON, "r", encoding="utf-8") as f:
    assigned_items = json.load(f)

art_to_cat = {}
for it in assigned_items:
    aid = extract_artwork_id(it)
    cat = extract_category(it)
    
    # ID와 카테고리가 모두 있어야 등록
    if aid and cat:
        art_to_cat[aid] = cat

print(f"[OK] Assignments loaded: {len(art_to_cat):,}")
print(f"   (Sample Assign IDs: {list(art_to_cat.keys())[:3]})")

# ==========================================
# 5. 조인 (Join)
# ==========================================
per_cat = {}
match_count = 0

for aid, cat in art_to_cat.items():
    if aid in vec_map:
        per_cat.setdefault(cat, []).append(vec_map[aid])
        match_count += 1

print(f"\n>>> [Result] Joined Categories: {len(per_cat)} categories")
print(f">>> [Result] Matched Vectors: {match_count:,} / {len(vec_map):,}")

if match_count == 0:
    print("🚨 여전히 매칭 실패! 키 이름을 다시 확인해보세요.")
    exit()

# ==========================================
# 6. Centroid 계산 및 저장
# ==========================================
centroid_records = []
counts = {}

for cat, vecs in per_cat.items():
    X = np.stack(vecs, axis=0)
    X = l2norm(X)
    n = X.shape[0]
    
    if n < max(MIN_K, 2):
        c = X.mean(axis=0)
    else:
        c0 = X.mean(axis=0)
        c0 = c0 / (np.linalg.norm(c0) + EPS)
        sims = X @ c0
        k = min(TOPK, n)
        top_idx = np.argsort(-sims)[:k]
        c = X[top_idx].mean(axis=0)
    
    c = c / (np.linalg.norm(c) + EPS)
    centroid_records.append({
        "category": cat,
        "centroid": c.tolist()
    })
    counts[cat] = n

OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_CENTROIDS_JSON, "w", encoding="utf-8") as f:
    json.dump(centroid_records, f, ensure_ascii=False)

print(f"\n🎉 저장 완료: {OUT_CENTROIDS_JSON}")

>>> Loading Vectors...
[OK] Vectors loaded: 27,702 (dim=512)
   (Sample Vector IDs: ['category030_1137', 'category030_0095', 'category030_1534'])
>>> Loading Assignments...
[OK] Assignments loaded: 27,702
   (Sample Assign IDs: ['category033_0001', 'category056_0001', 'category020_0001'])

>>> [Result] Joined Categories: 107 categories
>>> [Result] Matched Vectors: 27,702 / 27,702

🎉 저장 완료: /home/j-i14e107/Image_classification/outputs_json/category_centroids_topk.json


In [ ]:
# =========================================================
# EXPORT (patched): Standardized outputs for downstream
# - artwork_vector_norm.jsonl : {"artwork_id": <stem>, "vector": [512 floats]}
# - artwork_assigned_v2_post_singlecat.json :
#       ensure single category id in field "category_id" (string) and "has_category" (bool)
# =========================================================
import json
from pathlib import Path
import numpy as np

OUT_DIR = Path.cwd() / "outputs_json"
VEC_IN = OUT_DIR / "artwork_vector.json"
ASSIGN_IN = OUT_DIR / "artwork_assigned_v2_post.json"

VEC_OUT = OUT_DIR / "artwork_vector_norm.jsonl"
ASSIGN_OUT = OUT_DIR / "artwork_assigned_v2_post_singlecat.json"

def _stem(x):
    return Path(str(x)).stem

def load_vec_any(p: Path):
    raw = json.load(open(p, "r", encoding="utf-8"))
    out = {}
    if isinstance(raw, dict):
        for k,v in raw.items():
            if v is None:
                continue
            out[_stem(k)] = v
    elif isinstance(raw, list):
        for r in raw:
            if not isinstance(r, dict):
                continue
            aid = r.get("artwork_id") or r.get("id") or r.get("item_id")
            vec = r.get("vector") or r.get("artwork_vector") or r.get("embedding")
            if aid is None or vec is None:
                continue
            out[_stem(aid)] = vec
    else:
        raise ValueError("Unsupported vector format")
    return out

# 1) vectors -> normalized JSONL
vec_map = load_vec_any(VEC_IN)
# normalize (safeguard)
keys = list(vec_map.keys())
mat = np.asarray([vec_map[k] for k in keys], dtype=np.float32)
mat = mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)

with open(VEC_OUT, "w", encoding="utf-8") as f:
    for k, v in zip(keys, mat):
        f.write(json.dumps({"artwork_id": k, "vector": v.tolist()}, ensure_ascii=False) + "\n")

print("Saved:", VEC_OUT, "| rows:", len(keys), "| dim:", mat.shape[1])

# 2) assigned post -> single category field
assigned = json.load(open(ASSIGN_IN, "r", encoding="utf-8"))

def pick_first(d, keys):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None

def extract_single_category(a: dict):
    # Prefer explicit primary fields, otherwise first element of genre-like lists.
    v = pick_first(a, ["category_id", "primary_category", "primary_genre", "category", "label"])
    if v is None:
        v = pick_first(a, ["genre", "genres", "clip_genres"])
    if isinstance(v, list):
        v = v[0] if len(v) else None
    if v is None:
        return None
    return str(v)

out = []
missing = 0
for a in assigned:
    b = dict(a)  # shallow copy
    aid = pick_first(b, ["artwork_id", "id", "item_id"])
    if aid is not None:
        b["artwork_id"] = _stem(aid)
    cat = extract_single_category(b)
    if cat is None:
        b["category_id"] = None
        b["has_category"] = False
        missing += 1
    else:
        b["category_id"] = cat
        b["has_category"] = True
    out.append(b)

with open(ASSIGN_OUT, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print("Saved:", ASSIGN_OUT, "| rows:", len(out), "| missing_category:", missing)